<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/phase2_kvasir_capsule_data_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 2 - Build the raw medical data lake

Also called data-engineering and data-curation phase as we build a structured raw medical data lake from capsule-endoscopy data before creating visual artifacts or question-answer pairs. The primary dataset is Kvasir-Capsule, which contains capsule-endoscopy videos, labelled images, medically verified finding classes, bounding-box annotations, video identifiers, and frame numbers. The purpose of this phase is to transform the original dataset files into a clean, searchable, and reproducible project data layer.

### 1. Install dependencies and imports

In [ ]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    osfclient

In [ ]:
from pathlib import Path
from collections import defaultdict

import os
import re
import json
import random
import hashlib
import warnings
import subprocess
import shutil
from datetime import datetime, timezone

import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from pathlib import Path

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Load declarative configuration and reproducibility

In [ ]:
TIMEZONE = "America/Toronto"

RUN_ID = datetime.now(
    ZoneInfo(TIMEZONE)
).strftime(
    "%Y%m%d_%H%M%S"
)


CONFIG = {
    # --------------------------------------------------------------
    # Experiment identity
    # --------------------------------------------------------------

    "phase": "phase2",

    "experiment_name": (
        "phase2_kvasir_capsule_"
        "raw_medical_data_lake"
    ),

    "notebook_name": (
        "phase2_kvasir_capsule_"
        "data_lake.ipynb"
    ),

    "seed": 42,
    "run_id": RUN_ID,
    "timezone": TIMEZONE,

    # --------------------------------------------------------------
    # Storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    # --------------------------------------------------------------
    # Dataset source
    # --------------------------------------------------------------

    "dataset_name": "Kvasir-Capsule",
    "dataset_source": "OSF",
    "dataset_osf_project_id": "dv2ag",

    "dataset_download_enabled": False,

    # --------------------------------------------------------------
    # Raw dataset availability validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "required_metadata_file": ("metadata.csv"),
        "osf_storage_subdir": "osfstorage",

        # Official Kvasir-Capsule labelled-image directory.
        "labelled_images_subdir": "images",

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },

    # --------------------------------------------------------------
    # Expected dataset characteristics
    # --------------------------------------------------------------

    "expected_labelled_frames": 47238,
    "expected_classes": 14,

    "expected_labelled_videos": 43,

    "expected_unlabelled_videos": 74,

    "expected_total_videos": 117,

    "expected_total_extractable_frames": (
        4741504
    ),

    "expected_unlabelled_frames": (
        4694266
    ),

    # --------------------------------------------------------------
    # Video and frame alignment
    # --------------------------------------------------------------

    "original_capture_fps": 2.0,

    "expected_export_container_fps": 30.0,

    "frame_index_offset_candidates": [
        -1,
        0,
        1,
    ],

    "frame_alignment_sample_size": 30,
    "frame_alignment_min_valid_fraction": 0.80,

    "frame_alignment_comparison_size": [
        128,
        128,
    ],

    # --------------------------------------------------------------
    # Video-level split
    # --------------------------------------------------------------

    "split_strategy": (
        "video_level_multilabel_stratified"
    ),

    "train_fraction": 0.70,
    "validation_fraction": 0.15,
    "test_fraction": 0.15,

    # --------------------------------------------------------------
    # Verified finding segments
    # --------------------------------------------------------------

    "segment_max_gap_frames": 1,

    "minimum_verified_segment_frames": 2,

    # --------------------------------------------------------------
    # Domain adaptation
    # --------------------------------------------------------------

    "domain_adaptation_enabled": True,

    "domain_adaptation_include_fully_unlabelled_videos": (
        True
    ),

    "domain_adaptation_include_train_video_unlabelled_frames": (
        True
    ),

    "domain_adaptation_exclude_validation_videos": (
        True
    ),

    "domain_adaptation_exclude_test_videos": (
        True
    ),

    "domain_adaptation_sampling_seconds": 0.5,

    "domain_adaptation_extract_frames": False,

    # --------------------------------------------------------------
    # Temporal context
    # --------------------------------------------------------------

    "temporal_context_enabled": True,

    "temporal_context_offsets_seconds": [
        -2.0,
        -1.0,
        -0.5,
        0.0,
        0.5,
        1.0,
        2.0,
    ],

    "temporal_context_extract_frames": True,

    "temporal_context_image_format": "jpg",

    "temporal_context_jpeg_quality": 95,

    # --------------------------------------------------------------
    # Image quality control
    # --------------------------------------------------------------

    "qc_enabled": True,

    "qc_blur_quantile": 0.05,

    "qc_contrast_quantile": 0.05,

    "qc_brightness_low_quantile": 0.01,

    "qc_brightness_high_quantile": 0.99,

    # --------------------------------------------------------------
    # Normalized clinical taxonomy
    # --------------------------------------------------------------

    # Keys must match finding_class_normalized.
    "clinical_group_map": {
        "ampulla of vater": (
            "anatomical_landmark"
        ),

        "ileocecal valve": (
            "anatomical_landmark"
        ),

        "pylorus": (
            "anatomical_landmark"
        ),

        "normal clean mucosa": (
            "normal_mucosa"
        ),

        "reduced mucosal view": (
            "visibility_limitation"
        ),

        "blood fresh": "bleeding",

        "blood hematin": "bleeding",

        "angiectasia": (
            "vascular_lesion"
        ),

        "erosion": (
            "mucosal_lesion"
        ),

        "erythema": (
            "mucosal_lesion"
        ),

        "ulcer": (
            "mucosal_lesion"
        ),

        "lymphangiectasia": (
            "lymphatic_lesion"
        ),

        "polyp": (
            "protruding_lesion"
        ),

        "foreign body": (
            "foreign_body"
        ),
    },
}


CONFIG

In [ ]:

# Set random seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

### 3. Mount Google Drive Storage Backend

In [ ]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

### 4. Define data paths

In [ ]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS

### 5. Dataset verification

In [ ]:
def verify_raw_dataset(
    config,
    dirs,
):
    """
    Checks whether the minimum required raw Kvasir-Capsule
    files are available.

    Only the declared labelled-image directory is used when
    counting labelled images.
    """

    raw_data_dir = dirs["raw_data_dir"]
    validation = config[
        "dataset_validation"
    ]

    labelled_images_dir = (
        raw_data_dir
        / validation[
            "labelled_images_subdir"
        ]
    )

    if (
        not raw_data_dir.exists()
        or not labelled_images_dir.exists()
        or not metadata_path.is_file()
    ):
        return False

    metadata_available = any(
        raw_data_dir.rglob(
            validation[
                "required_metadata_file"
            ]
        )
    )

    video_count = sum(
        1
        for path in raw_data_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in validation[
                "video_extensions"
            ]
        )
    )

    labelled_image_count = sum(
        1
        for path
        in labelled_images_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in validation[
                "image_extensions"
            ]
        )
    )

    return (
        metadata_available
        and video_count
            >= validation[
                "minimum_video_files"
            ]
        and labelled_image_count
            >= validation[
                "minimum_labelled_images"
            ]
    )

RAW_DATASET_AVAILABLE = (
    verify_raw_dataset(
        config=CONFIG,
        dirs=DIRS,
    )
)

RAW_DATASET_AVAILABLE

In [ ]:
def download_dataset_if_needed(
    config,
    dirs,
):
    """
    Downloads Kvasir-Capsule when the minimum raw-dataset
    validation checks do not pass.

    Side effect:
        May download files into dirs["raw_data_dir"].

    Returns:
        "already_available" or "downloaded".
    """

    if verify_raw_dataset(config, dirs):
        print(
            "Raw-dataset validation passed. "
            "Skipping download."
        )
        return "already_available"

    if not config["dataset_download_enabled"]:
        raise RuntimeError(
            "Kvasir-Capsule is missing or incomplete, "
            "and automatic download is disabled."
        )

    if shutil.which("osf") is None:
        raise RuntimeError(
            "The 'osf' command is not installed or is not "
            "available on PATH. Install osfclient first."
        )

    raw_data_dir = dirs["raw_data_dir"]

    raw_data_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    command = [
        "osf",
        "-p",
        config["dataset_osf_project_id"],
        "clone",
        str(raw_data_dir),
    ]

    print(
        "Downloading Kvasir-Capsule to persistent storage..."
    )

    try:
        subprocess.run(
            command,
            check=True,
        )

    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "The OSF download command failed."
        ) from error

    if not verify_raw_dataset(config, dirs):
        raise RuntimeError(
            "The download command completed, but the "
            "downloaded dataset did not pass validation."
        )

    print(
        "Dataset download completed and validation passed."
    )

    return "downloaded"


DOWNLOAD_STATUS = download_dataset_if_needed(
    config=CONFIG,
    dirs=DIRS,
)

print("Download status:", DOWNLOAD_STATUS)

In [ ]:
# ------------------------------------------------------------------
# Raw-dataset file discovery
# ------------------------------------------------------------------

VALIDATION = CONFIG["dataset_validation"]

IMAGE_EXTENSIONS = frozenset(
    extension.casefold()
    for extension in VALIDATION[
        "image_extensions"
    ]
)

VIDEO_EXTENSIONS = frozenset(
    extension.casefold()
    for extension in VALIDATION[
        "video_extensions"
    ]
)


def discover_files(
    root,
    extensions,
):
    """
    Returns a deterministic collection of files whose
    extensions match the allowed extensions.

    The filesystem is inspected but not modified.
    """

    if not root.is_dir():
        raise NotADirectoryError(
            f"Dataset directory does not exist: {root}"
        )

    allowed_extensions = frozenset(
        extension.casefold()
        for extension in extensions
    )

    return tuple(
        sorted(
            path
            for path in root.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in allowed_extensions
            )
        )
    )

In [ ]:
if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        "The expected metadata file does not exist: "
        f"{METADATA_PATH}"
    )


df_raw = pd.read_csv(
    METADATA_PATH
)


print(
    "Discovered labelled images:",
    len(image_files),
)

print(
    "Discovered videos:",
    len(video_files),
)

print(
    "Metadata path:",
    METADATA_PATH,
)

print(
    "Metadata shape:",
    df_raw.shape,
)

display(
    df_raw.head()
)

print(
    "Raw metadata columns:",
    df_raw.columns.tolist(),
)

### 6. Normalization layer

In [ ]:
COLUMN_NAME_RULES = (
    (re.compile(r"[^a-z0-9]+"), "_"),
    (re.compile(r"_+"), "_"),
)

LABEL_RULES = (
    (re.compile(r"[_\-/]+"), " "),
    (re.compile(r"\s+"), " "),
)

COLUMN_ALIASES = {
    "file_name": "filename",
    "image_name": "filename",
    "image_filename": "filename",
    "video": "video_id",
    "video_name": "video_id",
    "frame": "frame_number",
    "frame_no": "frame_number",
    "label": "finding_class",
    "class": "finding_class",
    "category": "finding_category",
}

BBOX_SPEC = {
    "x_columns": ("x1", "x2", "x3", "x4"),
    "y_columns": ("y1", "y2", "y3", "y4"),
}

REQUIRED_COLUMNS = (
    "filename",
    "video_id",
    "finding_class",
)


def is_missing_scalar(value):
    """Returns True only for scalar missing values."""

    if value is None:
        return True

    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False

    return isinstance(
        missing,
        (bool, np.bool_),
    ) and bool(missing)


def scalar_to_text(value):
    """
    Converts a scalar metadata value to clean text.

    Sequences are rejected because Phase 2 metadata fields are
    expected to be scalar. This prevents silent label loss.
    """

    if isinstance(value, (list, tuple)):
        raise TypeError(
            "Expected a scalar metadata value, "
            f"but received {type(value).__name__}."
        )

    if is_missing_scalar(value):
        return ""

    return str(value).strip()


def apply_text_rules(
    text,
    rules,
):
    """Applies ordered regex transformations."""

    result = text

    for pattern, replacement in rules:
        result = pattern.sub(
            replacement,
            result,
        )

    return result


def apply_series_rules(
    series,
    rules,
):
    """Vectorized equivalent for a pandas Series."""

    result = series

    for pattern, replacement in rules:
        result = result.str.replace(
            pattern,
            replacement,
            regex=True,
        )

    return result


def normalize_column_name(value):
    """Creates a canonical snake_case column name."""

    text = scalar_to_text(value).casefold()

    return apply_text_rules(
        text=text,
        rules=COLUMN_NAME_RULES,
    ).strip("_")


def normalize_label_text(value):
    """Normalizes one scalar medical label."""

    text = (
        scalar_to_text(value)
        .casefold()
        .strip()
    )

    return apply_text_rules(
        text=text,
        rules=LABEL_RULES,
    ).strip()


def normalize_id(value):
    """Creates a filename-independent matching key."""

    text = Path(
        scalar_to_text(value)
    ).stem

    return re.sub(
        pattern=r"[^a-zA-Z0-9]+",
        repl="",
        string=text,
    ).casefold()

In [ ]:
def canonicalize_columns(
    dataframe,
    aliases,
):
    """
    Returns a new DataFrame with normalized and canonical columns.

    Raises an error if multiple source columns resolve to the
    same canonical column.
    """

    normalized_columns = [
        normalize_column_name(column)
        for column in dataframe.columns
    ]

    canonical_columns = [
        aliases.get(column, column)
        for column in normalized_columns
    ]

    canonical_index = pd.Index(
        canonical_columns
    )

    duplicate_columns = (
        canonical_index[
            canonical_index.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_columns:
        raise ValueError(
            "Multiple metadata columns resolve to the same "
            f"canonical name: {duplicate_columns}"
        )

    result = dataframe.copy()
    result.columns = canonical_columns

    return result


def validate_required_columns(
    dataframe,
    required_columns,
):
    """Validates the input schema without modifying it."""

    missing_columns = sorted(
        set(required_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Missing required metadata columns: "
            f"{missing_columns}"
        )

    return dataframe

In [ ]:
def add_label_columns(
    dataframe,
    clinical_group_map,
    strict=True,
):
    """
    Adds clean, normalized, and grouped label columns.

    The source label is preserved in finding_class_raw.
    """

    source_labels = (
        dataframe["finding_class_raw"]
        if "finding_class_raw" in dataframe.columns
        else dataframe["finding_class"]
    )

    clean_labels = source_labels.map(
        scalar_to_text
    )

    normalized_labels = apply_series_rules(
        series=(
            clean_labels
            .str.casefold()
            .str.strip()
        ),
        rules=LABEL_RULES,
    ).str.strip()

    clinical_groups = normalized_labels.map(
        clinical_group_map
    )

    unmapped_labels = sorted(
        normalized_labels[
            normalized_labels.ne("")
            & clinical_groups.isna()
        ]
        .unique()
        .tolist()
    )

    if strict and unmapped_labels:
        raise ValueError(
            "Labels missing from clinical_group_map: "
            f"{unmapped_labels}"
        )

    return dataframe.assign(
        finding_class_raw=source_labels,
        finding_class=clean_labels,
        finding_class_normalized=normalized_labels,
        clinical_group=clinical_groups,
    )


def add_matching_keys(dataframe):
    """Adds normalized image and video matching keys."""

    return dataframe.assign(
        image_key=(
            dataframe["filename"]
            .map(normalize_id)
        ),
        video_key=(
            dataframe["video_id"]
            .map(normalize_id)
        ),
    )

In [ ]:
def add_empty_bbox_columns(dataframe):
    """Returns the stable bbox schema for data without annotations."""

    index = dataframe.index

    return dataframe.assign(
        bbox_annotation_present=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_complete=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_invalid=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        has_bbox=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_xmin=np.nan,
        bbox_ymin=np.nan,
        bbox_xmax=np.nan,
        bbox_ymax=np.nan,
        bbox_width=np.nan,
        bbox_height=np.nan,
        bbox_area=np.nan,
    )


def normalize_bounding_boxes(
    dataframe,
    bbox_spec,
):
    """
    Adds a validated axis-aligned bounding-box representation.

    Original coordinate columns are not overwritten.
    """

    x_columns = tuple(
        bbox_spec["x_columns"]
    )

    y_columns = tuple(
        bbox_spec["y_columns"]
    )

    expected_columns = (
        x_columns
        + y_columns
    )

    present_columns = tuple(
        column
        for column in expected_columns
        if column in dataframe.columns
    )

    if not present_columns:
        return add_empty_bbox_columns(
            dataframe
        )

    missing_schema_columns = sorted(
        set(expected_columns)
        - set(present_columns)
    )

    if missing_schema_columns:
        raise ValueError(
            "Incomplete bounding-box schema. "
            f"Missing columns: {missing_schema_columns}"
        )

    coordinates = (
        dataframe
        .loc[:, expected_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    annotation_present = (
        coordinates
        .notna()
        .any(axis=1)
    )

    bbox_complete = (
        coordinates
        .notna()
        .all(axis=1)
    )

    xmin = coordinates[
        list(x_columns)
    ].min(axis=1)

    xmax = coordinates[
        list(x_columns)
    ].max(axis=1)

    ymin = coordinates[
        list(y_columns)
    ].min(axis=1)

    ymax = coordinates[
        list(y_columns)
    ].max(axis=1)

    geometry_valid = (
        bbox_complete
        & xmax.gt(xmin)
        & ymax.gt(ymin)
    )

    width = (
        xmax - xmin
    ).where(geometry_valid)

    height = (
        ymax - ymin
    ).where(geometry_valid)

    return dataframe.assign(
        bbox_annotation_present=annotation_present,
        bbox_complete=bbox_complete,
        bbox_invalid=(
            annotation_present
            & ~geometry_valid
        ),
        has_bbox=geometry_valid,
        bbox_xmin=xmin.where(geometry_valid),
        bbox_ymin=ymin.where(geometry_valid),
        bbox_xmax=xmax.where(geometry_valid),
        bbox_ymax=ymax.where(geometry_valid),
        bbox_width=width,
        bbox_height=height,
        bbox_area=width * height,
    )

In [ ]:
def normalize_metadata(
    dataframe,
    clinical_group_map,
):
    """
    Pure-by-contract Phase 2 metadata normalization pipeline.

    Input:
        raw DataFrame

    Output:
        new normalized DataFrame
    """

    return (
        dataframe
        .pipe(
            canonicalize_columns,
            aliases=COLUMN_ALIASES,
        )
        .pipe(
            validate_required_columns,
            required_columns=REQUIRED_COLUMNS,
        )
        .pipe(
            add_label_columns,
            clinical_group_map=clinical_group_map,
            strict=True,
        )
        .pipe(
            add_matching_keys,
        )
        .pipe(
            normalize_bounding_boxes,
            bbox_spec=BBOX_SPEC,
        )
    )


df = normalize_metadata(
    dataframe=df_raw,
    clinical_group_map=(
        CONFIG["clinical_group_map"]
    ),
)

In [ ]:
# ------------------------------------------------------------------
# Ground-truth label accessors
# ------------------------------------------------------------------

def get_gold_label(example):
    """
    Returns the cleaned dataset-provided ground-truth label
    from a normalized metadata example.
    """

    return example["finding_class"]


def get_normalized_gold_label(example):
    """
    Returns the normalized dataset-provided ground-truth label
    from a normalized metadata example.
    """

    return example[
        "finding_class_normalized"
    ]

### 7. Dataset validation

In [ ]:
def validate_dataset_characteristics(
    dataframe,
    config,
):
    """
    Validates key Kvasir-Capsule characteristics against
    the expected values declared in CONFIG.

    This is a sanity check for dataset integrity and
    reproducibility. It does not modify the data.
    """

    checks = {
        "labelled_frames": (
            len(dataframe),
            config["expected_labelled_frames"],
        ),

        "finding_classes": (
            dataframe["finding_class"].nunique(),
            config["expected_classes"],
        ),

        "labelled_videos": (
            dataframe["video_key"].nunique(),
            config["expected_labelled_videos"],
        ),
    }

    for name, (actual, expected) in checks.items():

        if actual != expected:

            warnings.warn(
                f"{name}: expected {expected}, "
                f"found {actual}"
            )

        else:

            print(
                f"PASS: {name} = {actual}"
            )

validate_dataset_characteristics(
    df,
    CONFIG,
)

### 8. Map labelled images and videos to files



In [ ]:
def build_file_index(
    files,
):
    """
    Builds a lookup index from normalized file IDs
    to physical file paths.
    """

    index = defaultdict(list)

    for path in files:
        index[
            normalize_id(path.name)
        ].append(path)

    return index


def resolve_unique_file(
    key,
    file_index,
):
    """
    Returns the file path when exactly one file
    matches the normalized identifier.
    """

    matches = file_index.get(
        key,
        [],
    )

    return (
        matches[0]

        if len(matches) == 1
        else None
    )

image_index = build_file_index(
    image_files
)


df["image_path"] = (
    df["image_key"]
    .map(
        lambda key: resolve_unique_file(
            key,
            image_index,
        )
    )
)


df["image_exists"] = (
    df["image_path"]
    .notna()
)

video_index = build_file_index(
    video_files
)


df["video_path"] = (
    df["video_key"]
    .map(
        lambda key: resolve_unique_file(
            key,
            video_index,
        )
    )
)


df["video_exists"] = (
    df["video_path"]
    .notna()
)

print(
    "Metadata rows with resolved video:",
    df["video_exists"].sum(),
    "/",
    len(df),
)

print(
    "Unique metadata rows with resolved video:",
    df.loc[
        df["video_exists"],
        "video_key",
    ].nunique(),
)


def find_index_collisions(file_index):
    """
    Returns normalized IDs associated with multiple files.
    """

    return {
        key: tuple(paths)
        for key, paths in file_index.items()
        if len(paths) > 1
    }


image_collisions = find_index_collisions(
    image_index
)

video_collisions = find_index_collisions(
    video_index
)


if image_collisions:
    raise ValueError(
        "Ambiguous normalized image IDs found: "
        f"{list(image_collisions)[:10]}"
    )

if video_collisions:
    raise ValueError(
        "Ambiguous normalized video IDs found: "
        f"{list(video_collisions)[:10]}"
    )

### 9. Build video manifest

In [ ]:
def build_video_record(
    video_path,
    labelled_video_keys,
    dataset_root,
):
    """
    Builds one structured manifest record for a video file.

    Reads technical metadata from the video without modifying
    the source file.
    """

    video_key = normalize_id(
        video_path.name
    )

    video_info = probe_video(
        video_path
    )

    is_labelled_video = (
        video_key
        in labelled_video_keys
    )

    return {
        **video_info,

        "video_key":
            video_key,

        "video_filename":
            video_path.name,

        "video_path":
            str(video_path),

        "video_relative_path":
            video_path
            .relative_to(dataset_root)
            .as_posix(),

        "video_annotation_type": (
            "partially_labelled"
            if is_labelled_video
            else "fully_unlabelled"
        ),
    }


labelled_video_keys = frozenset(
    key
    for key in (
        df["video_key"]
        .dropna()
        .unique()
    )
    if key != ""
)


video_manifest = pd.DataFrame.from_records(
    build_video_record(
        video_path=video_path,
        labelled_video_keys=(
            labelled_video_keys
        ),
        dataset_root=(
            DIRS["dataset_root_dir"]
        ),
    )
    for video_path in tqdm(
        video_files,
        desc="Building video manifest",
    )
)


display(
    video_manifest.head()
)

In [ ]:
def validate_video_manifest(
    dataframe,
    labelled_video_keys,
    config,
):
    """
    Validates the physical video inventory and its consistency
    with the labelled metadata.

    Does not modify the DataFrame.
    """

    required_columns = {
        "video_key",
        "video_filename",
        "video_path",
        "video_relative_path",
        "video_annotation_type",
    }

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Video manifest is missing required columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------------
    # Validate normalized physical video identifiers
    # --------------------------------------------------------------

    if dataframe["video_key"].eq("").any():
        raise ValueError(
            "Video manifest contains empty video keys."
        )

    duplicate_video_keys = (
        dataframe.loc[
            dataframe["video_key"]
            .duplicated(keep=False),
            "video_key",
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_keys:
        raise ValueError(
            "Multiple physical videos resolve to the same "
            f"video key: {duplicate_video_keys}"
        )


    # --------------------------------------------------------------
    # Validate the complete physical video inventory
    # --------------------------------------------------------------

    actual_total_videos = len(
        dataframe
    )

    expected_total_videos = config[
        "expected_total_videos"
    ]

    if (
        actual_total_videos
        != expected_total_videos
    ):
        raise ValueError(
            "Unexpected number of physical videos: "
            f"expected {expected_total_videos}, "
            f"found {actual_total_videos}"
        )


    # --------------------------------------------------------------
    # Validate annotation categories
    # --------------------------------------------------------------

    expected_annotation_types = {
        "partially_labelled",
        "fully_unlabelled",
    }

    actual_annotation_types = set(
        dataframe[
            "video_annotation_type"
        ]
        .dropna()
        .unique()
    )

    unexpected_annotation_types = sorted(
        actual_annotation_types
        - expected_annotation_types
    )

    if unexpected_annotation_types:
        raise ValueError(
            "Unexpected video annotation types: "
            f"{unexpected_annotation_types}"
        )


    # --------------------------------------------------------------
    # Cross-check metadata against physical videos
    # --------------------------------------------------------------

    manifest_labelled_video_keys = frozenset(
        dataframe.loc[
            dataframe[
                "video_annotation_type"
            ].eq("partially_labelled"),
            "video_key",
        ]
    )

    metadata_labelled_video_keys = frozenset(
        labelled_video_keys
    )

    missing_physical_videos = sorted(
        metadata_labelled_video_keys
        - manifest_labelled_video_keys
    )

    unexpected_labelled_videos = sorted(
        manifest_labelled_video_keys
        - metadata_labelled_video_keys
    )

    if (
        missing_physical_videos
        or unexpected_labelled_videos
    ):
        raise ValueError(
            "Video manifest is inconsistent with metadata. "
            f"Missing physical labelled videos: "
            f"{missing_physical_videos}. "
            f"Unexpected labelled videos: "
            f"{unexpected_labelled_videos}."
        )

    return dataframe

video_manifest = (
    video_manifest
    .pipe(
        validate_video_manifest,
        labelled_video_keys=(
            labelled_video_keys
        ),
        config=CONFIG,
    )
)

video_counts = (
    video_manifest[
        "video_annotation_type"
    ]
    .value_counts()
)

print(
    "Total physical videos:",
    len(video_manifest),
)

print(
    video_counts
)

### 10. Probe video metadata

In [ ]:
def empty_video_probe():
    """
    Returns the stable schema for an unreadable video.
    """

    return {
        "container_opened": False,
        "first_frame_readable": False,
        "container_fps": np.nan,
        "frame_count": np.nan,
        "width": np.nan,
        "height": np.nan,
        "container_duration_seconds": np.nan,
    }


def probe_video(
    video_path,
):
    """
    Reads technical video metadata without modifying the file.

    The reported FPS and duration describe the exported video
    container, not necessarily the original clinical timeline.
    """

    cap = cv2.VideoCapture(
        str(video_path)
    )

    try:
        if not cap.isOpened():
            return empty_video_probe()

        raw_fps = cap.get(
            cv2.CAP_PROP_FPS
        )

        raw_frame_count = cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )

        raw_width = cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )

        raw_height = cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )

        first_frame_readable, _ = (
            cap.read()
        )

    finally:
        cap.release()


    fps = (
        float(raw_fps)
        if (
            np.isfinite(raw_fps)
            and raw_fps > 0
        )
        else np.nan
    )

    frame_count = (
        int(raw_frame_count)
        if (
            np.isfinite(raw_frame_count)
            and raw_frame_count > 0
        )
        else np.nan
    )

    width = (
        int(raw_width)
        if (
            np.isfinite(raw_width)
            and raw_width > 0
        )
        else np.nan
    )

    height = (
        int(raw_height)
        if (
            np.isfinite(raw_height)
            and raw_height > 0
        )
        else np.nan
    )

    container_duration_seconds = (
        frame_count / fps
        if (
            np.isfinite(frame_count)
            and np.isfinite(fps)
        )
        else np.nan
    )


    return {
        "container_opened": True,

        "first_frame_readable": bool(
            first_frame_readable
        ),

        "container_fps":
            fps,

        "frame_count":
            frame_count,

        "width":
            width,

        "height":
            height,

        "container_duration_seconds":
            container_duration_seconds,
    }

### 11. Verify frame-number alignment

In [ ]:
# ------------------------------------------------------------------
# Video-frame alignment
# ------------------------------------------------------------------

def read_video_frame(
    video_path,
    frame_index,
):
    """
    Reads one OpenCV-indexed frame from a video.

    Returns:
        BGR image array when successful.
        None when the frame cannot be read.
    """

    try:
        frame_index = int(
            frame_index
        )

    except (TypeError, ValueError):
        return None

    if frame_index < 0:
        return None

    cap = cv2.VideoCapture(
        str(video_path)
    )

    try:
        if not cap.isOpened():
            return None

        seek_succeeded = cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_index,
        )

        if not seek_succeeded:
            return None

        success, frame = cap.read()

        return (
            frame
            if success
            else None
        )

    finally:
        cap.release()


def image_similarity(
    image_a,
    image_b,
    comparison_size,
):
    """
    Computes grayscale structural similarity between two
    images after resizing them to a common resolution.
    """

    if image_a is None or image_b is None:
        return np.nan

    comparison_size = tuple(
        int(value)
        for value in comparison_size
    )

    if (
        len(comparison_size) != 2
        or any(
            value <= 0
            for value in comparison_size
        )
    ):
        raise ValueError(
            "comparison_size must contain two positive "
            f"integers, found: {comparison_size}"
        )

    gray_a = cv2.cvtColor(
        image_a,
        cv2.COLOR_BGR2GRAY,
    )

    gray_b = cv2.cvtColor(
        image_b,
        cv2.COLOR_BGR2GRAY,
    )

    resized_a = cv2.resize(
        gray_a,
        comparison_size,
    )

    resized_b = cv2.resize(
        gray_b,
        comparison_size,
    )

    return float(
        ssim(
            resized_a,
            resized_b,
            data_range=255,
        )
    )


def infer_frame_index_offset(
    dataframe,
    config,
):
    """
    Infers the alignment between Kvasir metadata frame numbers
    and zero-based OpenCV video-frame indices.

    One candidate frame is sampled from each selected video
    to avoid overrepresenting videos with many labelled frames.

    Returns:
        selected_offset
        alignment_scores
    """

    required_columns = {
        "image_exists",
        "video_exists",
        "image_path",
        "video_path",
        "video_key",
        "frame_number",
    }

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Frame-alignment input is missing columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------------
    # Validate and normalize frame numbers
    # --------------------------------------------------------------

    frame_numbers = pd.to_numeric(
        dataframe["frame_number"],
        errors="coerce",
    )

    valid_frame_numbers = (
        frame_numbers.notna()
        & frame_numbers.ge(0)
        & frame_numbers.eq(
            np.floor(frame_numbers)
        )
    )

    eligible_rows = (
        dataframe["image_exists"]
        & dataframe["video_exists"]
        & valid_frame_numbers
    )

    candidates = (
        dataframe.loc[
            eligible_rows
        ]
        .assign(
            alignment_frame_number=(
                frame_numbers.loc[
                    eligible_rows
                ]
                .astype("int64")
            )
        )
    )

    if candidates.empty:
        raise RuntimeError(
            "No valid labelled image/video pairs are available "
            "for frame-alignment verification."
        )


    # --------------------------------------------------------------
    # Sample across distinct labelled videos
    # --------------------------------------------------------------

    shuffled_candidates = candidates.sample(
        frac=1,
        random_state=config["seed"],
    )

    unique_video_candidates = (
        shuffled_candidates
        .drop_duplicates(
            subset="video_key"
        )
    )

    sample_size = min(
        config["frame_alignment_sample_size"],
        len(unique_video_candidates),
    )

    sampled_examples = (
        unique_video_candidates
        .head(sample_size)
    )

    if sampled_examples.empty:
        raise RuntimeError(
            "No frame-alignment examples could be sampled."
        )


    # --------------------------------------------------------------
    # Evaluate configured candidate offsets
    # --------------------------------------------------------------

    candidate_offsets = tuple(
        int(offset)
        for offset in config[
            "frame_index_offset_candidates"
        ]
    )

    if not candidate_offsets:
        raise ValueError(
            "No frame-index offset candidates are configured."
        )

    comparison_size = config[
        "frame_alignment_comparison_size"
    ]

    alignment_scores = {}

    for offset in candidate_offsets:
        scores = []

        for _, example in (
            sampled_examples.iterrows()
        ):
            labelled_image = cv2.imread(
                str(example["image_path"])
            )

            video_frame = read_video_frame(
                video_path=example["video_path"],
                frame_index=(
                    example[
                        "alignment_frame_number"
                    ]
                    + offset
                ),
            )

            score = image_similarity(
                image_a=labelled_image,
                image_b=video_frame,
                comparison_size=comparison_size,
            )

            if np.isfinite(score):
                scores.append(
                    float(score)
                )

        alignment alignment_scores[offset] = {
            "mean_ssim": (
                float(np.mean(scores))
                if scores
                else np.nan
            ),

            "median_ssim": (
                float(np.median(scores))
                if scores
                else np.nan
            ),

            "valid_pairs":
                len(scores),

            "sampled_pairs":
                len(sampled_examples),
        }


    # --------------------------------------------------------------
    # Select the best sufficiently supported offset
    # --------------------------------------------------------------

    minimum_valid_fraction = float(
        config[
            "frame_alignment_min_valid_fraction"
        ]
    )

    if not (
        0 < minimum_valid_fraction <= 1
    ):
        raise ValueError(
            "frame_alignment_min_valid_fraction must be "
            "within the interval (0, 1]."
        )

    minimum_valid_pairs = max(
        1,
        int(
            np.ceil(
                minimum_valid_fraction
                * len(sampled_examples)
            )
        ),
    )

    valid_scores = {
        offset: statistics["mean_ssim"]
        for offset, statistics
        in alignment_scores.items()
        if (
            statistics["valid_pairs"]
            >= minimum_valid_pairs
            and np.isfinite(
                statistics["mean_ssim"]
            )
        )
    }

    if not valid_scores:
        raise RuntimeError(
            "Frame alignment could not be evaluated with "
            "enough valid image/video pairs for any offset."
        )

    selected_offset = max(
        valid_scores,
        key=valid_scores.get,
    )

    return (
        selected_offset,
        alignment_scores,
    )


FRAME_INDEX_OFFSET, FRAME_ALIGNMENT_SCORES = (
    infer_frame_index_offset(
        dataframe=df,
        config=CONFIG,
    )
)


frame_alignment_report = (
    pd.DataFrame
    .from_dict(
        FRAME_ALIGNMENT_SCORES,
        orient="index",
    )
    .rename_axis(
        "offset"
    )
    .reset_index()
    .sort_values(
        "mean_ssim",
        ascending=False,
    )
)


display(
    frame_alignment_report
)

print(
    "Selected frame index offset:",
    FRAME_INDEX_OFFSET,
)

### 12. Audit clinical label taxonomy





In [ ]:
# ------------------------------------------------------------------
# Clinical label taxonomy audit
# ------------------------------------------------------------------

clinical_taxonomy_report = (
    df[
        [
            "finding_class",
            "finding_class_normalized",
            "clinical_group",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "clinical_group",
            "finding_class_normalized",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "PASS: all non-empty finding classes were mapped "
    "to a clinical group during metadata normalization."
)

print(
    "Finding classes:",
    clinical_taxonomy_report[
        "finding_class_normalized"
    ].nunique(),
)

print(
    "Clinical groups:",
    clinical_taxonomy_report[
        "clinical_group"
    ].nunique(),
)

display(
    clinical_taxonomy_report
)

### 13. Build labelled-frame manifest

In [ ]:
# ------------------------------------------------------------------
# Portable path handling
# ------------------------------------------------------------------

def path_relative_to_storage(
    value,
    storage_root,
):
    """
    Converts a physical path into a POSIX path relative to
    storage_root.

    Raises an error when the physical path is outside the
    configured storage root.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None

    except TypeError:
        pass

    storage_root = Path(
        storage_root
    ).resolve()

    path = Path(
        value
    ).resolve()

    try:
        return (
            path
            .relative_to(storage_root)
            .as_posix()
        )

    except ValueError as error:
        raise ValueError(
            "Manifest path is outside storage_root: "
            f"{path}"
        ) from error


# ------------------------------------------------------------------
# Labelled-frame manifest
# ------------------------------------------------------------------

def build_labelled_frame_manifest(
    dataframe,
    frame_index_offset,
    storage_root,
    bbox_spec,
):
    """
    Builds the trusted labelled-frame manifest.

    One row represents one medically verified
    Kvasir-Capsule frame.

    Returns a new DataFrame.
    """

    base_columns = [
        # Image identity and location
        "filename",
        "image_key",
        "image_path",
        "image_exists",

        # Video identity and location
        "video_id",
        "video_key",
        "video_path",
        "video_exists",

        # Temporal ground truth
        "frame_number",

        # Medical ground truth
        "finding_class_raw",
        "finding_class",
        "finding_class_normalized",
        "clinical_group",

        # Bounding-box state
        "bbox_annotation_present",
        "bbox_complete",
        "bbox_invalid",
        "has_bbox",

        # Standardized bounding box
        "bbox_xmin",
        "bbox_ymin",
        "bbox_xmax",
        "bbox_ymax",
        "bbox_width",
        "bbox_height",
        "bbox_area",
    ]

    optional_columns = [
        column
        for column in (
            "finding_category",
            *bbox_spec["x_columns"],
            *bbox_spec["y_columns"],
        )
        if column in dataframe.columns
    ]

    selected_columns = (
        base_columns
        + optional_columns
    )


    # --------------------------------------------------------------
    # Validate required columns
    # --------------------------------------------------------------

    missing_columns = sorted(
        set(base_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Labelled-frame manifest is missing columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------------
    # Validate physical file mappings
    # --------------------------------------------------------------

    unresolved_images = (
        ~dataframe["image_exists"]
        .fillna(False)
    )

    unresolved_videos = (
        ~dataframe["video_exists"]
        .fillna(False)
    )

    if unresolved_images.any():
        raise ValueError(
            "Cannot build trusted manifest: "
            f"{int(unresolved_images.sum())} "
            "labelled images are unresolved."
        )

    if unresolved_videos.any():
        raise ValueError(
            "Cannot build trusted manifest: "
            f"{int(unresolved_videos.sum())} "
            "source videos are unresolved."
        )


    # --------------------------------------------------------------
    # Validate frame numbers
    # --------------------------------------------------------------

    frame_numbers = pd.to_numeric(
        dataframe["frame_number"],
        errors="coerce",
    )

    valid_frame_numbers = (
        frame_numbers.notna()
        & frame_numbers.ge(0)
        & frame_numbers.eq(
            np.floor(frame_numbers)
        )
    )

    if not valid_frame_numbers.all():
        raise ValueError(
            "Cannot build trusted manifest: "
            f"{int((~valid_frame_numbers).sum())} "
            "invalid frame numbers were found."
        )

    frame_numbers = (
        frame_numbers
        .astype("Int64")
    )

    frame_index_offset = int(
        frame_index_offset
    )

    opencv_frame_indices = (
        frame_numbers
        + frame_index_offset
    )

    if opencv_frame_indices.lt(0).any():
        raise ValueError(
            "The selected frame-index offset produces "
            "negative OpenCV frame indices."
        )


    # --------------------------------------------------------------
    # Build immutable-by-contract result
    # --------------------------------------------------------------

    result = (
        dataframe[
            selected_columns
        ]
        .copy()
        .assign(
            frame_number=frame_numbers,

            frame_index_offset=(
                frame_index_offset
            ),

            opencv_frame_index=(
                opencv_frame_indices
            ),

            image_relpath=(
                dataframe["image_path"]
                .map(
                    lambda value:
                    path_relative_to_storage(
                        value=value,
                        storage_root=storage_root,
                    )
                )
            ),

            video_relpath=(
                dataframe["video_path"]
                .map(
                    lambda value:
                    path_relative_to_storage(
                        value=value,
                        storage_root=storage_root,
                    )
                )
            ),

            annotation_status=(
                "medically_verified"
            ),

            label_source=(
                "kvasir_capsule_expert_annotation"
            ),
        )
    )

    return result


labelled_frame_manifest = (
    build_labelled_frame_manifest(
        dataframe=df,
        frame_index_offset=(
            FRAME_INDEX_OFFSET
        ),
        storage_root=CONFIG[
            "storage_root"
        ],
        bbox_spec=BBOX_SPEC,
    )
)


print(
    "Labelled-frame manifest shape:",
    labelled_frame_manifest.shape,
)

display(
    labelled_frame_manifest.head()
)

### 14. Create video/class matrix

In [ ]:
def build_video_class_matrix(
    labelled_manifest,
    expected_classes,
):
    """
    Builds a stable video-level multilabel presence matrix.

    Rows:
        Labelled videos.

    Columns:
        Normalized Kvasir finding classes.

    Values:
        1 when at least one verified frame of the class
        appears in the video; otherwise 0.
    """

    required_columns = {
        "video_key",
        "finding_class_normalized",
    }

    missing_columns = sorted(
        required_columns
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build video-class matrix. "
            f"Missing columns: {missing_columns}"
        )

    video_keys = (
        labelled_manifest["video_key"]
        .astype("string")
        .str.strip()
    )

    finding_classes = (
        labelled_manifest[
            "finding_class_normalized"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_rows = (
        video_keys.isna()
        | video_keys.eq("")
        | finding_classes.isna()
        | finding_classes.eq("")
    )

    if invalid_rows.any():
        raise ValueError(
            "Cannot build video-class matrix: "
            f"{int(invalid_rows.sum())} rows contain "
            "missing video keys or finding classes."
        )

    expected_classes = tuple(
        sorted(
            expected_classes
        )
    )

    observed_classes = set(
        finding_classes.unique()
    )

    missing_classes = sorted(
        set(expected_classes)
        - observed_classes
    )

    unexpected_classes = sorted(
        observed_classes
        - set(expected_classes)
    )

    if missing_classes or unexpected_classes:
        raise ValueError(
            "Video-class taxonomy mismatch. "
            f"Missing classes: {missing_classes}. "
            f"Unexpected classes: {unexpected_classes}."
        )

    matrix = pd.crosstab(
        video_keys,
        finding_classes,
    )

    return (
        matrix
        .gt(0)
        .astype("int8")
        .reindex(
            columns=expected_classes,
            fill_value=0,
        )
        .rename_axis(
            index="video_key",
            columns="finding_class_normalized",
        )
    )

video_class_matrix = (
    build_video_class_matrix(
        labelled_manifest=(
            labelled_frame_manifest
        ),
        expected_classes=(
            CONFIG[
                "clinical_group_map"
            ].keys()
        ),
    )
)


video_class_support = (
    video_class_matrix
    .sum(axis=0)
    .sort_values()
    .rename(
        "labelled_video_count"
    )
    .to_frame()
)


print(
    "Video-class matrix shape:",
    video_class_matrix.shape,
)

display(
    video_class_matrix.head()
)

display(
    video_class_support
)

### 15. Multilabel-stratified video split

In [ ]:
def validate_split_configuration(
    config,
):
    """
    Validates and normalizes the configured video-level split.

    Returns normalized split fractions and random seed.
    """

    required_keys = {
        "train_fraction",
        "validation_fraction",
        "test_fraction",
        "split_strategy",
        "seed",
    }

    missing_keys = sorted(
        required_keys
        - set(config)
    )

    if missing_keys:
        raise KeyError(
            "Split configuration is missing keys: "
            f"{missing_keys}"
        )

    fractions = {
        "train": float(
            config["train_fraction"]
        ),
        "validation": float(
            config["validation_fraction"]
        ),
        "test": float(
            config["test_fraction"]
        ),
    }

    invalid_fractions = {
        name: value
        for name, value in fractions.items()
        if not 0.0 < value < 1.0
    }

    if invalid_fractions:
        raise ValueError(
            "Every split fraction must be between "
            f"0 and 1: {invalid_fractions}"
        )

    if not np.isclose(
        sum(fractions.values()),
        1.0,
    ):
        raise ValueError(
            "train_fraction + validation_fraction + "
            "test_fraction must equal 1.0."
        )

    expected_strategy = (
        "video_level_multilabel_stratified"
    )

    if (
        config["split_strategy"]
        != expected_strategy
    ):
        raise ValueError(
            "Unsupported split strategy: "
            f"{config['split_strategy']}"
        )

    try:
        seed = int(
            config["seed"]
        )

    except (TypeError, ValueError) as error:
        raise ValueError(
            "The split seed must be an integer."
        ) from error

    return {
        **fractions,
        "seed": seed,
    }


def split_labelled_videos(
    video_class_matrix,
    config,
):
    """
    Splits labelled videos using multilabel stratification.

    Returns sets of video keys for train, validation, and test.
    """

    split_config = (
        validate_split_configuration(
            config
        )
    )

    if video_class_matrix.empty:
        raise ValueError(
            "Cannot split an empty video-class matrix."
        )

    if not video_class_matrix.index.is_unique:
        raise ValueError(
            "Video-class matrix contains duplicate video keys."
        )

    if video_class_matrix.isna().any().any():
        raise ValueError(
            "Video-class matrix contains missing label values."
        )

    valid_label_values = (
        video_class_matrix
        .isin(
            [0, 1]
        )
        .all()
        .all()
    )

    if not valid_label_values:
        raise ValueError(
            "Video-class matrix must contain only binary "
            "label values: 0 or 1."
        )

    video_keys = (
        video_class_matrix
        .index
        .to_numpy()
    )

    labels = (
        video_class_matrix
        .to_numpy(
            dtype=np.int8
        )
    )

    dummy_features = np.zeros(
        (
            len(video_keys),
            1,
        ),
        dtype=np.int8,
    )

    temporary_fraction = (
        split_config["validation"]
        + split_config["test"]
    )

    first_splitter = (
        MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=temporary_fraction,
            random_state=split_config["seed"],
        )
    )

    train_indices, temporary_indices = next(
        first_splitter.split(
            dummy_features,
            labels,
        )
    )

    relative_test_fraction = (
        split_config["test"]
        / temporary_fraction
    )

    second_splitter = (
        MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=relative_test_fraction,
            random_state=split_config["seed"],
        )
    )

    validation_relative, test_relative = next(
        second_splitter.split(
            dummy_features[
                temporary_indices
            ],
            labels[
                temporary_indices
            ],
        )
    )

    validation_indices = (
        temporary_indices[
            validation_relative
        ]
    )

    test_indices = (
        temporary_indices[
            test_relative
        ]
    )

    return {
        "train": set(
            video_keys[
                train_indices
            ]
        ),
        "validation": set(
            video_keys[
                validation_indices
            ]
        ),
        "test": set(
            video_keys[
                test_indices
            ]
        ),
    }


VIDEO_SPLITS = split_labelled_videos(
    video_class_matrix=video_class_matrix,
    config=CONFIG,
)

VIDEO_SPLITS

### 16. Propagate split



In [ ]:
def validate_video_splits(
    video_splits,
    expected_video_keys,
):
    """
    Validates that labelled videos form a complete,
    non-overlapping train/validation/test partition.
    """

    required_split_names = {
        "train",
        "validation",
        "test",
    }

    actual_split_names = set(
        video_splits
    )

    missing_split_names = sorted(
        required_split_names
        - actual_split_names
    )

    unexpected_split_names = sorted(
        actual_split_names
        - required_split_names
    )

    if (
        missing_split_names
        or unexpected_split_names
    ):
        raise KeyError(
            "Invalid video split names. "
            f"Missing: {missing_split_names}. "
            f"Unexpected: {unexpected_split_names}."
        )

    train = set(
        video_splits["train"]
    )

    validation = set(
        video_splits["validation"]
    )

    test = set(
        video_splits["test"]
    )

    empty_splits = [
        split_name
        for split_name, video_keys
        in {
            "train": train,
            "validation": validation,
            "test": test,
        }.items()
        if not video_keys
    ]

    if empty_splits:
        raise RuntimeError(
            "Empty video splits found: "
            f"{empty_splits}"
        )

    overlaps = {
        "train_validation": (
            train
            & validation
        ),
        "train_test": (
            train
            & test
        ),
        "validation_test": (
            validation
            & test
        ),
    }

    nonempty_overlaps = {
        name: sorted(video_keys)
        for name, video_keys
        in overlaps.items()
        if video_keys
    }

    if nonempty_overlaps:
        raise RuntimeError(
            "Video leakage detected between splits: "
            f"{nonempty_overlaps}"
        )

    expected_video_keys = set(
        expected_video_keys
    )

    assigned_video_keys = (
        train
        | validation
        | test
    )

    missing_video_keys = sorted(
        expected_video_keys
        - assigned_video_keys
    )

    unexpected_video_keys = sorted(
        assigned_video_keys
        - expected_video_keys
    )

    if (
        missing_video_keys
        or unexpected_video_keys
    ):
        raise RuntimeError(
            "Video split coverage mismatch. "
            f"Missing videos: {missing_video_keys}. "
            f"Unexpected videos: {unexpected_video_keys}."
        )


def build_split_lookup(
    video_splits,
):
    """
    Creates a video_key-to-split mapping.
    """

    return {
        video_key: split_name
        for split_name, video_keys
        in video_splits.items()
        for video_key in video_keys
    }


validate_video_splits(
    video_splits=VIDEO_SPLITS,
    expected_video_keys=(
        video_class_matrix.index
    ),
)


SPLIT_LOOKUP = build_split_lookup(
    VIDEO_SPLITS
)


labelled_frame_manifest = (
    labelled_frame_manifest
    .assign(
        split=lambda data:
            data["video_key"]
            .map(SPLIT_LOOKUP)
    )
)


unassigned_frame_count = int(
    labelled_frame_manifest[
        "split"
    ]
    .isna()
    .sum()
)

if unassigned_frame_count:
    raise RuntimeError(
        "Some labelled frames did not receive a split: "
        f"{unassigned_frame_count}"
    )

### 17. Assign roles to all videos

In [ ]:
def derive_video_role(
    annotation_type,
    supervised_split,
    config,
):
    """
    Determines the validated Phase 2 role of one video.
    """

    if pd.isna(annotation_type):
        raise ValueError(
            "Video annotation type is missing."
        )

    annotation_type = str(
        annotation_type
    )

    has_supervised_split = (
        pd.notna(supervised_split)
    )

    if annotation_type == "fully_unlabelled":

        if has_supervised_split:
            raise ValueError(
                "A fully unlabelled video cannot belong "
                "to a supervised split."
            )

        return (
            "domain_adaptation_only"
            if config[
                "domain_adaptation_include_fully_unlabelled_videos"
            ]
            else "unlabelled_reserved"
        )

    if annotation_type != "partially_labelled":
        raise ValueError(
            "Unsupported video annotation type: "
            f"{annotation_type}"
        )

    if not has_supervised_split:
        raise ValueError(
            "A partially labelled video did not receive "
            "a supervised split."
        )

    if supervised_split == "train":
        return (
            "supervised_train_plus_domain_adaptation"
            if config[
                "domain_adaptation_include_train_video_unlabelled_frames"
            ]
            else "supervised_train"
        )

    if supervised_split == "validation":
        return "supervised_validation"

    if supervised_split == "test":
        return "supervised_test"

    raise ValueError(
        "Unsupported supervised split: "
        f"{supervised_split}"
    )


def assign_video_roles(
    video_manifest,
    split_lookup,
    config,
    storage_root,
):
    """
    Adds the supervised split and validated Phase 2
    data role to every video.

    Returns a new DataFrame.
    """

    required_columns = {
        "video_key",
        "video_path",
        "video_annotation_type",
    }

    missing_columns = sorted(
        required_columns
        - set(video_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot assign video roles. "
            f"Missing columns: {missing_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    result = (
        video_manifest
        .copy()
        .assign(
            supervised_split=lambda data:
                data["video_key"]
                .map(split_lookup),

            video_relpath=lambda data:
                data["video_path"]
                .map(
                    lambda value:
                        path_relative_to_storage(
                            value=value,
                            storage_root=storage_root,
                        )
                ),
        )
    )

    result["data_role"] = [
        derive_video_role(
            annotation_type=annotation_type,
            supervised_split=supervised_split,
            config=config,
        )
        for annotation_type, supervised_split
        in zip(
            result["video_annotation_type"],
            result["supervised_split"],
        )
    ]

    return result


video_manifest = assign_video_roles(
    video_manifest=video_manifest,
    split_lookup=SPLIT_LOOKUP,
    config=CONFIG,
    storage_root=CONFIG[
        "storage_root"
    ],
)


video_role_report = (
    video_manifest[
        [
            "video_annotation_type",
            "supervised_split",
            "data_role",
        ]
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "video_count"
    )
    .reset_index()
)


display(
    video_role_report
)

### 18. Identify domain-adaptation source videos

In [ ]:
def build_verified_frame_lookup(
    labelled_manifest,
):
    """
    Returns verified OpenCV frame indices grouped by video.

    Used to prevent medically labelled frames from entering
    the unlabelled domain-adaptation pool.
    """

    return (
        labelled_manifest

        .groupby(
            "video_key"
        )[
            "opencv_frame_index"
        ]

        .agg(
            lambda values:
                frozenset(
                    values
                    .dropna()
                    .astype(int)
                )
        )

        .to_dict()
    )


def select_domain_adaptation_videos(
    video_manifest,
):
    """
    Selects videos whose configured data role allows them
    to contribute unlabelled frames to domain adaptation.
    """

    allowed_roles = {
        "domain_adaptation_only",
        "supervised_train_plus_domain_adaptation",
    }

    return (
        video_manifest[
            video_manifest[
                "data_role"
            ].isin(
                allowed_roles
            )
        ]
        .copy()
    )


VERIFIED_FRAME_LOOKUP = (
    build_verified_frame_lookup(
        labelled_frame_manifest
    )
)


domain_source_videos = (
    select_domain_adaptation_videos(
        video_manifest
    )
)


print(
    "Domain-adaptation source videos:",
    len(domain_source_videos),
)

### 19. Generate domain-adaptation references

In [ ]:
def get_domain_sampling_stride(
    config,
    video_fps,
):
    """
    Converts the configured sampling interval in seconds
    into frame units for one source video.
    """

    sampling_seconds = float(
        config[
            "domain_adaptation_sampling_seconds"
        ]
    )

    video_fps = float(
        video_fps
    )

    if (
        not np.isfinite(sampling_seconds)
        or sampling_seconds <= 0
    ):
        raise ValueError(
            "Domain-adaptation sampling seconds must "
            "be a positive finite number."
        )

    if (
        not np.isfinite(video_fps)
        or video_fps <= 0
    ):
        raise ValueError(
            "Video FPS must be a positive finite number."
        )

    return max(
        1,
        int(
            round(
                sampling_seconds
                * video_fps
            )
        ),
    )


def build_domain_records_for_video(
    video,
    verified_frame_lookup,
    config,
    frame_index_offset,
):
    """
    Builds unlabelled frame references for one eligible video.

    verified_frame_lookup must contain OpenCV frame indices.

    No images are extracted.
    No pseudo-labels are created.
    """

    required_fields = {
        "video_key",
        "video_relpath",
        "video_annotation_type",
        "supervised_split",
        "data_role",
        "frame_count",
        "fps",
    }

    missing_fields = sorted(
        required_fields
        - set(video.index)
    )

    if missing_fields:
        raise KeyError(
            "Cannot build domain-adaptation records. "
            f"Missing video fields: {missing_fields}"
        )

    annotation_type = str(
        video["video_annotation_type"]
    )

    supervised_split = (
        None
        if pd.isna(
            video["supervised_split"]
        )
        else str(
            video["supervised_split"]
        )
    )

    data_role = str(
        video["data_role"]
    )

    is_fully_unlabelled_source = (
        annotation_type
        == "fully_unlabelled"
        and supervised_split is None
        and data_role
        == "domain_adaptation_only"
    )

    is_train_unlabelled_source = (
        annotation_type
        == "partially_labelled"
        and supervised_split
        == "train"
        and data_role
        == "supervised_train_plus_domain_adaptation"
    )

    if not (
        is_fully_unlabelled_source
        or is_train_unlabelled_source
    ):
        raise ValueError(
            "Video is not eligible for domain adaptation: "
            f"{video['video_key']}"
        )

    frame_count_value = float(
        video["frame_count"]
    )

    if (
        not np.isfinite(frame_count_value)
        or frame_count_value <= 0
        or not frame_count_value.is_integer()
    ):
        raise ValueError(
            "Invalid frame count for video: "
            f"{video['video_key']}"
        )

    frame_count = int(
        frame_count_value
    )

    video_fps = float(
        video["fps"]
    )

    stride = get_domain_sampling_stride(
        config=config,
        video_fps=video_fps,
    )

    opencv_indices = np.arange(
        0,
        frame_count,
        stride,
        dtype=np.int64,
    )

    verified_indices = np.asarray(
        tuple(
            verified_frame_lookup.get(
                video["video_key"],
                frozenset(),
            )
        ),
        dtype=np.int64,
    )

    if verified_indices.size:

        opencv_indices = opencv_indices[
            ~np.isin(
                opencv_indices,
                verified_indices,
            )
        ]

    frame_index_offset = int(
        frame_index_offset
    )

    metadata_frame_numbers = (
        opencv_indices
        - frame_index_offset
    )

    valid_mask = (
        metadata_frame_numbers
        >= 0
    )

    opencv_indices = opencv_indices[
        valid_mask
    ]

    metadata_frame_numbers = (
        metadata_frame_numbers[
            valid_mask
        ]
    )

    source_type = (
        "fully_unlabelled_video"
        if is_fully_unlabelled_source
        else "unlabelled_frame_from_train_video"
    )

    return pd.DataFrame(
        {
            "video_key":
                video["video_key"],

            "video_relpath":
                video["video_relpath"],

            "opencv_frame_index":
                opencv_indices,

            "frame_number":
                metadata_frame_numbers,

            "source_timestamp_seconds":
                (
                    opencv_indices
                    / video_fps
                ),

            "source_type":
                source_type,

            "annotation_status":
                "unlabelled",

            "ground_truth_label":
                pd.NA,

            "supervised_split":
                (
                    supervised_split
                    if supervised_split is not None
                    else pd.NA
                ),

            "purpose":
                "visual_domain_adaptation",
        }
    )

### 20. Build domain-adaptation manifest

In [ ]:
DOMAIN_ADAPTATION_COLUMNS = [
    "video_key",
    "video_relpath",
    "opencv_frame_index",
    "frame_number",
    "source_timestamp_seconds",
    "source_type",
    "annotation_status",
    "ground_truth_label",
    "supervised_split",
    "purpose",
]


def build_domain_adaptation_manifest(
    domain_source_videos,
    verified_frame_lookup,
    config,
    frame_index_offset,
):
    """
    Builds the complete domain-adaptation reference manifest.

    One row represents one unlabelled frame reference.
    No images are extracted or saved.
    """

    required_columns = {
        "video_key",
        "video_relpath",
        "video_annotation_type",
        "supervised_split",
        "data_role",
        "frame_count",
        "fps",
    }

    missing_columns = sorted(
        required_columns
        - set(domain_source_videos.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build domain-adaptation manifest. "
            f"Missing columns: {missing_columns}"
        )

    if not domain_source_videos[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Domain source videos contain duplicate "
            "video keys."
        )

    eligible_roles = {
        "domain_adaptation_only",
        "supervised_train_plus_domain_adaptation",
    }

    invalid_role_mask = (
        ~domain_source_videos[
            "data_role"
        ]
        .isin(
            eligible_roles
        )
    )

    if invalid_role_mask.any():

        invalid_videos = (
            domain_source_videos.loc[
                invalid_role_mask,
                [
                    "video_key",
                    "supervised_split",
                    "data_role",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Non-eligible videos were included in the "
            "domain-adaptation source pool: "
            f"{invalid_videos}"
        )

    if domain_source_videos.empty:
        return pd.DataFrame(
            columns=DOMAIN_ADAPTATION_COLUMNS
        )

    domain_tables = [
        build_domain_records_for_video(
            video=video,
            verified_frame_lookup=(
                verified_frame_lookup
            ),
            config=config,
            frame_index_offset=(
                frame_index_offset
            ),
        )
        for _, video
        in tqdm(
            domain_source_videos.iterrows(),
            total=len(
                domain_source_videos
            ),
            desc=(
                "Building domain-adaptation manifest"
            ),
        )
    ]

    non_empty_tables = [
        table
        for table in domain_tables
        if not table.empty
    ]

    if not non_empty_tables:
        return pd.DataFrame(
            columns=DOMAIN_ADAPTATION_COLUMNS
        )

    result = (
        pd.concat(
            non_empty_tables,
            ignore_index=True,
        )
        .loc[
            :,
            DOMAIN_ADAPTATION_COLUMNS,
        ]
        .sort_values(
            [
                "video_key",
                "opencv_frame_index",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    duplicate_reference_mask = (
        result.duplicated(
            subset=[
                "video_key",
                "opencv_frame_index",
            ],
            keep=False,
        )
    )

    if duplicate_reference_mask.any():

        duplicate_references = (
            result.loc[
                duplicate_reference_mask,
                [
                    "video_key",
                    "opencv_frame_index",
                ],
            ]
            .drop_duplicates()
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Duplicate domain-adaptation frame "
            "references found: "
            f"{duplicate_references}"
        )

    return result


domain_adaptation_manifest = (
    build_domain_adaptation_manifest(
        domain_source_videos=(
            domain_source_videos
        ),
        verified_frame_lookup=(
            VERIFIED_FRAME_LOOKUP
        ),
        config=CONFIG,
        frame_index_offset=(
            FRAME_INDEX_OFFSET
        ),
    )
)


domain_adaptation_summary = (
    domain_adaptation_manifest
    .groupby(
        "source_type",
        dropna=False,
    )
    .size()
    .rename(
        "frame_reference_count"
    )
    .reset_index()
)


print(
    "Domain-adaptation frame references:",
    f"{len(domain_adaptation_manifest):,}",
)

display(
    domain_adaptation_summary
)

display(
    domain_adaptation_manifest.head()
)

### 21. Build verified finding segments

In [ ]:
def add_verified_segment_numbers(
    labelled_manifest,
    config,
):
    """
    Groups temporally adjacent verified frames of the same
    finding within the same video.

    Returns a new DataFrame with a segment_number column.
    """

    group_columns = [
        "video_key",
        "finding_class_normalized",
    ]

    required_columns = {
        *group_columns,
        "frame_number",
    }

    missing_columns = sorted(
        required_columns
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build verified segments. "
            f"Missing columns: {missing_columns}"
        )

    max_gap = config[
        "segment_max_gap_frames"
    ]

    if (
        isinstance(max_gap, bool)
        or not isinstance(
            max_gap,
            (int, np.integer),
        )
        or max_gap < 1
    ):
        raise ValueError(
            "segment_max_gap_frames must be "
            "a positive integer."
        )

    result = (
        labelled_manifest
        .sort_values(
            group_columns
            + ["frame_number"],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    frame_gap = (
        result
        .groupby(
            group_columns,
            sort=False,
            observed=True,
        )[
            "frame_number"
        ]
        .diff()
    )

    starts_new_segment = (
        frame_gap.isna()
        | frame_gap.gt(
            max_gap
        )
    )

    segment_numbers = (
        starts_new_segment
        .astype("int8")
        .groupby(
            [
                result["video_key"],
                result[
                    "finding_class_normalized"
                ],
            ],
            sort=False,
            observed=True,
        )
        .cumsum()
        .astype("Int64")
    )

    return result.assign(
        segment_number=segment_numbers
    )


labelled_frame_manifest = (
    add_verified_segment_numbers(
        labelled_manifest=(
            labelled_frame_manifest
        ),
        config=CONFIG,
    )
)

### 22. Create stable segment IDs

In [ ]:
def normalize_identifier_series(
    values,
):
    """
    Converts a pandas Series into stable identifier
    components using vectorized string operations.
    """

    return (
        values
        .astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(
            r"[^a-z0-9]+",
            "_",
            regex=True,
        )
        .str.strip("_")
    )


def add_finding_segment_ids(
    labelled_manifest,
):
    """
    Creates stable IDs for verified finding segments.

    Every frame belonging to the same verified segment
    receives the same finding_segment_id.
    """

    identity_columns = [
        "video_key",
        "finding_class_normalized",
        "segment_number",
    ]

    missing_columns = sorted(
        set(identity_columns)
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot create finding segment IDs. "
            f"Missing columns: {missing_columns}"
        )

    result = (
        labelled_manifest
        .copy()
    )

    video_ids = (
        normalize_identifier_series(
            result["video_key"]
        )
    )

    class_ids = (
        normalize_identifier_series(
            result[
                "finding_class_normalized"
            ]
        )
    )

    segment_numbers = pd.to_numeric(
        result["segment_number"],
        errors="coerce",
    )

    invalid_video_ids = (
        video_ids.isna()
        | video_ids.eq("")
    )

    invalid_class_ids = (
        class_ids.isna()
        | class_ids.eq("")
    )

    invalid_segment_numbers = (
        segment_numbers.isna()
        | segment_numbers.lt(1)
        | segment_numbers.mod(1).ne(0)
    )

    if invalid_video_ids.any():
        raise ValueError(
            "Cannot create finding segment IDs: "
            f"{int(invalid_video_ids.sum())} rows have "
            "invalid video keys."
        )

    if invalid_class_ids.any():
        raise ValueError(
            "Cannot create finding segment IDs: "
            f"{int(invalid_class_ids.sum())} rows have "
            "invalid finding classes."
        )

    if invalid_segment_numbers.any():
        raise ValueError(
            "Cannot create finding segment IDs: "
            f"{int(invalid_segment_numbers.sum())} rows "
            "have invalid segment numbers."
        )

    segment_number_ids = (
        segment_numbers
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

    finding_segment_ids = (
        video_ids
        .str.cat(
            class_ids,
            sep="__",
        )
        .str.cat(
            segment_number_ids,
            sep="__",
        )
    )

    result = result.assign(
        segment_number=(
            segment_numbers
            .astype("Int64")
        ),
        finding_segment_id=(
            finding_segment_ids
        ),
    )

    segment_identity = (
        result[
            identity_columns
            + ["finding_segment_id"]
        ]
        .drop_duplicates()
    )

    collision_mask = (
        segment_identity[
            "finding_segment_id"
        ]
        .duplicated(
            keep=False
        )
    )

    if collision_mask.any():

        collisions = (
            segment_identity.loc[
                collision_mask
            ]
            .sort_values(
                "finding_segment_id"
            )
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Finding segment ID collisions detected: "
            f"{collisions}"
        )

    return result


labelled_frame_manifest = (
    add_finding_segment_ids(
        labelled_manifest=(
            labelled_frame_manifest
        )
    )
)

### 23. Build finding-segment manifest

In [ ]:
SEGMENT_MANIFEST_COLUMNS = [
    "finding_segment_id",
    "video_key",
    "finding_class",
    "finding_class_normalized",
    "clinical_group",
    "verified_start_frame",
    "verified_end_frame",
    "verified_frame_count",
    "target_frame_number",
    "target_opencv_frame_index",
    "target_filename",
    "target_image_path",
    "target_image_relpath",
    "target_has_bbox",
    "target_bbox_xmin",
    "target_bbox_ymin",
    "target_bbox_xmax",
    "target_bbox_ymax",
    "split",
]


def build_finding_segment_manifest(
    labelled_manifest,
):
    """
    Aggregates verified frame-level annotations into
    one record per finding segment.

    The representative target is the middle verified
    frame in temporal order.
    """

    required_columns = {
        "finding_segment_id",
        "video_key",
        "finding_class",
        "finding_class_normalized",
        "clinical_group",
        "frame_number",
        "opencv_frame_index",
        "filename",
        "image_path",
        "image_relpath",
        "has_bbox",
        "bbox_xmin",
        "bbox_ymin",
        "bbox_xmax",
        "bbox_ymax",
        "split",
    }

    missing_columns = sorted(
        required_columns
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build finding segment manifest. "
            f"Missing columns: {missing_columns}"
        )

    if labelled_manifest.empty:
        return pd.DataFrame(
            columns=SEGMENT_MANIFEST_COLUMNS
        )

    invalid_segment_id_mask = (
        labelled_manifest[
            "finding_segment_id"
        ]
        .astype("string")
        .str.strip()
        .isna()
        |
        labelled_manifest[
            "finding_segment_id"
        ]
        .astype("string")
        .str.strip()
        .eq("")
    )

    if invalid_segment_id_mask.any():
        raise ValueError(
            "Finding segment manifest contains "
            f"{int(invalid_segment_id_mask.sum())} "
            "rows without a valid segment ID."
        )

    result = (
        labelled_manifest
        .sort_values(
            [
                "finding_segment_id",
                "frame_number",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    duplicate_frame_mask = (
        result.duplicated(
            subset=[
                "finding_segment_id",
                "frame_number",
            ],
            keep=False,
        )
    )

    if duplicate_frame_mask.any():

        duplicate_frames = (
            result.loc[
                duplicate_frame_mask,
                [
                    "finding_segment_id",
                    "frame_number",
                ],
            ]
            .drop_duplicates()
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Duplicate verified frames found within "
            f"finding segments: {duplicate_frames}"
        )

    invariant_columns = [
        "video_key",
        "finding_class",
        "finding_class_normalized",
        "clinical_group",
        "split",
    ]

    invariant_counts = (
        result
        .groupby(
            "finding_segment_id",
            sort=False,
            observed=True,
        )[
            invariant_columns
        ]
        .nunique(
            dropna=False
        )
    )

    inconsistent_segment_mask = (
        invariant_counts
        .gt(1)
        .any(axis=1)
    )

    if inconsistent_segment_mask.any():

        inconsistent_segment_ids = (
            invariant_counts.index[
                inconsistent_segment_mask
            ]
            .tolist()
        )

        raise ValueError(
            "Inconsistent metadata found within "
            "finding segments: "
            f"{inconsistent_segment_ids}"
        )

    segment_group = (
        result.groupby(
            "finding_segment_id",
            sort=False,
            observed=True,
        )
    )

    segment_positions = (
        segment_group
        .cumcount()
    )

    segment_sizes = (
        segment_group[
            "frame_number"
        ]
        .transform(
            "size"
        )
    )

    target_mask = (
        segment_positions
        .eq(
            segment_sizes
            .floordiv(2)
        )
    )

    segment_summary = (
        segment_group
        .agg(
            video_key=(
                "video_key",
                "first",
            ),
            finding_class=(
                "finding_class",
                "first",
            ),
            finding_class_normalized=(
                "finding_class_normalized",
                "first",
            ),
            clinical_group=(
                "clinical_group",
                "first",
            ),
            verified_start_frame=(
                "frame_number",
                "min",
            ),
            verified_end_frame=(
                "frame_number",
                "max",
            ),
            verified_frame_count=(
                "frame_number",
                "size",
            ),
            split=(
                "split",
                "first",
            ),
        )
        .reset_index()
    )

    target_frames = (
        result.loc[
            target_mask,
            [
                "finding_segment_id",
                "frame_number",
                "opencv_frame_index",
                "filename",
                "image_path",
                "image_relpath",
                "has_bbox",
                "bbox_xmin",
                "bbox_ymin",
                "bbox_xmax",
                "bbox_ymax",
            ],
        ]
        .rename(
            columns={
                "frame_number":
                    "target_frame_number",

                "opencv_frame_index":
                    "target_opencv_frame_index",

                "filename":
                    "target_filename",

                "image_path":
                    "target_image_path",

                "image_relpath":
                    "target_image_relpath",

                "has_bbox":
                    "target_has_bbox",

                "bbox_xmin":
                    "target_bbox_xmin",

                "bbox_ymin":
                    "target_bbox_ymin",

                "bbox_xmax":
                    "target_bbox_xmax",

                "bbox_ymax":
                    "target_bbox_ymax",
            }
        )
    )

    finding_segment_manifest = (
        segment_summary
        .merge(
            target_frames,
            on="finding_segment_id",
            how="left",
            validate="one_to_one",
        )
        .loc[
            :,
            SEGMENT_MANIFEST_COLUMNS,
        ]
        .sort_values(
            [
                "video_key",
                "verified_start_frame",
                "finding_class_normalized",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return finding_segment_manifest


finding_segment_manifest = (
    build_finding_segment_manifest(
        labelled_manifest=(
            labelled_frame_manifest
        )
    )
)


print(
    "Verified finding segments:",
    f"{len(finding_segment_manifest):,}",
)

display(
    finding_segment_manifest.head()
)

### 24. Determine temporal-grounding supervision

In [ ]:
def add_temporal_supervision_metadata(
    segment_manifest,
    config,
):
    """
    Describes the verified temporal supervision available
    for each finding segment.

    Observed segment limits are not treated as exact clinical
    onset or offset boundaries.
    """

    required_columns = {
        "verified_start_frame",
        "verified_end_frame",
        "verified_frame_count",
    }

    missing_columns = sorted(
        required_columns
        - set(segment_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot add temporal supervision metadata. "
            f"Missing columns: {missing_columns}"
        )

    minimum_frames = config[
        "minimum_verified_segment_frames"
    ]

    if (
        isinstance(minimum_frames, bool)
        or not isinstance(
            minimum_frames,
            (int, np.integer),
        )
        or minimum_frames < 2
    ):
        raise ValueError(
            "minimum_verified_segment_frames must be "
            "an integer greater than or equal to 2."
        )

    verified_frame_counts = pd.to_numeric(
        segment_manifest[
            "verified_frame_count"
        ],
        errors="coerce",
    )

    verified_start_frames = pd.to_numeric(
        segment_manifest[
            "verified_start_frame"
        ],
        errors="coerce",
    )

    verified_end_frames = pd.to_numeric(
        segment_manifest[
            "verified_end_frame"
        ],
        errors="coerce",
    )

    invalid_count_mask = (
        verified_frame_counts.isna()
        | verified_frame_counts.lt(1)
        | verified_frame_counts.mod(1).ne(0)
    )

    invalid_range_mask = (
        verified_start_frames.isna()
        | verified_end_frames.isna()
        | verified_start_frames.lt(0)
        | verified_end_frames.lt(
            verified_start_frames
        )
    )

    if invalid_count_mask.any():
        raise ValueError(
            "Invalid verified frame counts found: "
            f"{int(invalid_count_mask.sum())}"
        )

    if invalid_range_mask.any():
        raise ValueError(
            "Invalid verified temporal ranges found: "
            f"{int(invalid_range_mask.sum())}"
        )

    has_temporal_extent = (
        verified_end_frames
        .gt(
            verified_start_frames
        )
    )

    temporal_grounding_candidate = (
        verified_frame_counts
        .ge(
            minimum_frames
        )
        & has_temporal_extent
    )

    temporal_supervision_level = np.select(
        [
            temporal_grounding_candidate,
            verified_frame_counts.eq(1),
        ],
        [
            "multi_frame_weak_supervision",
            "single_frame_anchor_only",
        ],
        default=(
            "insufficient_multi_frame_supervision"
        ),
    )

    return (
        segment_manifest
        .assign(
            verified_frame_count=(
                verified_frame_counts
                .astype("Int64")
            ),

            temporal_grounding_candidate=(
                temporal_grounding_candidate
                .astype("boolean")
            ),

            temporal_supervision_level=(
                pd.Series(
                    temporal_supervision_level,
                    index=segment_manifest.index,
                    dtype="string",
                )
            ),

            # Kvasir-Capsule provides verified positive
            # frames, not exact clinical onset/offset.
            verified_temporal_boundary_available=False,
        )
    )


finding_segment_manifest = (
    add_temporal_supervision_metadata(
        segment_manifest=(
            finding_segment_manifest
        ),
        config=CONFIG,
    )
)

### 25. Build temporal evidence windows

In [ ]:
TEMPORAL_MANIFEST_COLUMNS = [
    "finding_segment_id",
    "video_key",
    "video_relpath",
    "finding_class",
    "finding_class_normalized",
    "split",
    "target_frame_number",
    "target_opencv_frame_index",
    "temporal_offset_seconds",
    "effective_temporal_offset_seconds",
    "frame_delta",
    "context_frame_number",
    "opencv_frame_index",
    "is_target",
    "requested_context_count",
    "available_context_count",
    "temporal_window_complete",
]


def build_temporal_manifest(
    segment_manifest,
    video_manifest,
    config,
    frame_index_offset,
):
    """
    Builds temporal evidence references around eligible
    verified finding segments.

    One row represents one valid temporal context frame.
    No images are extracted.
    """

    required_segment_columns = {
        "finding_segment_id",
        "video_key",
        "finding_class",
        "finding_class_normalized",
        "split",
        "target_frame_number",
        "target_opencv_frame_index",
        "temporal_grounding_candidate",
    }

    required_video_columns = {
        "video_key",
        "video_relpath",
        "frame_count",
        "fps",
    }

    missing_segment_columns = sorted(
        required_segment_columns
        - set(segment_manifest.columns)
    )

    missing_video_columns = sorted(
        required_video_columns
        - set(video_manifest.columns)
    )

    if missing_segment_columns:
        raise KeyError(
            "Temporal manifest is missing segment columns: "
            f"{missing_segment_columns}"
        )

    if missing_video_columns:
        raise KeyError(
            "Temporal manifest is missing video columns: "
            f"{missing_video_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    candidate_flags = (
        segment_manifest[
            "temporal_grounding_candidate"
        ]
        .astype("boolean")
    )

    if candidate_flags.isna().any():
        raise ValueError(
            "Temporal-grounding candidate values "
            "contain missing entries."
        )

    eligible_segments = (
        segment_manifest.loc[
            candidate_flags
        ]
        .copy()
    )

    if eligible_segments.empty:
        return pd.DataFrame(
            columns=TEMPORAL_MANIFEST_COLUMNS
        )

    raw_offsets = pd.Series(
        config[
            "temporal_context_offsets_seconds"
        ],
        dtype="object",
    )

    temporal_offsets = pd.to_numeric(
        raw_offsets,
        errors="coerce",
    )

    if (
        temporal_offsets.empty
        or temporal_offsets.isna().any()
        or not np.isfinite(
            temporal_offsets
        ).all()
    ):
        raise ValueError(
            "Temporal context offsets must be "
            "finite numeric values."
        )

    temporal_offsets = (
        temporal_offsets
        .astype("float64")
    )

    temporal_offsets = temporal_offsets.mask(
        np.isclose(
            temporal_offsets,
            0.0,
        ),
        0.0,
    )

    if temporal_offsets.duplicated().any():
        raise ValueError(
            "Temporal context offsets contain "
            "duplicate values."
        )

    if not temporal_offsets.eq(0.0).any():
        raise ValueError(
            "Temporal context offsets must include 0 "
            "for the target frame."
        )

    offset_table = (
        pd.DataFrame(
            {
                "temporal_offset_seconds":
                    temporal_offsets
            }
        )
        .sort_values(
            "temporal_offset_seconds"
        )
        .reset_index(
            drop=True
        )
    )

    video_metadata = (
        video_manifest[
            [
                "video_key",
                "video_relpath",
                "frame_count",
                "fps",
            ]
        ]
        .copy()
        .assign(
            frame_count=lambda data:
                pd.to_numeric(
                    data["frame_count"],
                    errors="coerce",
                ),

            fps=lambda data:
                pd.to_numeric(
                    data["fps"],
                    errors="coerce",
                ),
        )
    )

    invalid_video_metadata = (
        video_metadata["frame_count"].isna()
        | video_metadata["frame_count"].le(0)
        | video_metadata["frame_count"].mod(1).ne(0)
        | video_metadata["fps"].isna()
        | video_metadata["fps"].le(0)
        | ~np.isfinite(
            video_metadata["fps"]
        )
    )

    if invalid_video_metadata.any():

        invalid_video_keys = (
            video_metadata.loc[
                invalid_video_metadata,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid video frame count or FPS for: "
            f"{invalid_video_keys}"
        )

    video_metadata = (
        video_metadata
        .assign(
            frame_count=lambda data:
                data["frame_count"]
                .astype("Int64")
        )
    )

    segment_video_metadata = (
        eligible_segments
        .merge(
            video_metadata,
            on="video_key",
            how="left",
            validate="many_to_one",
            indicator=True,
        )
    )

    missing_video_mask = (
        segment_video_metadata[
            "_merge"
        ]
        .ne("both")
    )

    if missing_video_mask.any():

        missing_video_keys = (
            segment_video_metadata.loc[
                missing_video_mask,
                "video_key",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise KeyError(
            "Segment videos are missing from the "
            f"video manifest: {missing_video_keys}"
        )

    segment_video_metadata = (
        segment_video_metadata
        .drop(
            columns="_merge"
        )
    )

    expanded = (
        segment_video_metadata
        .merge(
            offset_table,
            how="cross",
        )
    )

    expanded = (
        expanded
        .assign(
            frame_delta=lambda data:
                (
                    data[
                        "temporal_offset_seconds"
                    ]
                    * data["fps"]
                )
                .round()
                .astype("Int64")
        )
    )

    duplicate_frame_delta_mask = (
        expanded.duplicated(
            subset=[
                "finding_segment_id",
                "frame_delta",
            ],
            keep=False,
        )
    )

    if duplicate_frame_delta_mask.any():

        duplicate_offsets = (
            expanded.loc[
                duplicate_frame_delta_mask,
                [
                    "finding_segment_id",
                    "temporal_offset_seconds",
                    "frame_delta",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Different temporal offsets resolve to "
            "the same frame: "
            f"{duplicate_offsets}"
        )

    frame_index_offset = int(
        frame_index_offset
    )

    expanded = (
        expanded
        .assign(
            opencv_frame_index=lambda data:
                (
                    data[
                        "target_opencv_frame_index"
                    ]
                    + data["frame_delta"]
                )
                .astype("Int64"),

            effective_temporal_offset_seconds=lambda data:
                (
                    data["frame_delta"]
                    / data["fps"]
                ),
        )
        .assign(
            context_frame_number=lambda data:
                (
                    data["opencv_frame_index"]
                    - frame_index_offset
                )
                .astype("Int64"),

            is_target=lambda data:
                data[
                    "temporal_offset_seconds"
                ]
                .eq(0.0),
        )
    )

    expanded = (
        expanded
        .assign(
            context_available=lambda data:
                (
                    data["opencv_frame_index"].ge(0)
                    & data["opencv_frame_index"].lt(
                        data["frame_count"]
                    )
                    & data[
                        "context_frame_number"
                    ].ge(0)
                )
        )
    )

    window_statistics = (
        expanded
        .groupby(
            "finding_segment_id",
            sort=False,
            observed=True,
        )
        .agg(
            requested_context_count=(
                "context_available",
                "size",
            ),
            available_context_count=(
                "context_available",
                "sum",
            ),
            temporal_window_complete=(
                "context_available",
                "all",
            ),
        )
        .reset_index()
    )

    temporal_manifest = (
        expanded.loc[
            expanded[
                "context_available"
            ]
        ]
        .merge(
            window_statistics,
            on="finding_segment_id",
            how="left",
            validate="many_to_one",
        )
        .loc[
            :,
            TEMPORAL_MANIFEST_COLUMNS,
        ]
        .sort_values(
            [
                "finding_segment_id",
                "temporal_offset_seconds",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return temporal_manifest


temporal_manifest = (
    build_temporal_manifest(
        segment_manifest=(
            finding_segment_manifest
        ),
        video_manifest=video_manifest,
        config=CONFIG,
        frame_index_offset=(
            FRAME_INDEX_OFFSET
        ),
    )
)


temporal_window_report = (
    temporal_manifest[
        [
            "finding_segment_id",
            "temporal_window_complete",
        ]
    ]
    .drop_duplicates()
)


print(
    "Temporal evidence frame references:",
    f"{len(temporal_manifest):,}",
)

print(
    "Temporal evidence windows:",
    f"{len(temporal_window_report):,}",
)

print(
    "Incomplete temporal windows:",
    int(
        (
            ~temporal_window_report[
                "temporal_window_complete"
            ]
        )
        .sum()
    ),
)


display(
    temporal_manifest.head()
)

### 26. Determine whether each temporal frame is verified or unknown

In [ ]:
def add_temporal_context_metadata(
    temporal_manifest,
    labelled_manifest,
):
    """
    Adds annotation status and image-source metadata
    to temporal context-frame references.

    An unlabelled context means that no ground-truth
    annotation is available. It does not mean normal.
    """

    required_temporal_columns = {
        "video_key",
        "context_frame_number",
        "finding_class_normalized",
        "is_target",
    }

    required_labelled_columns = {
        "video_key",
        "frame_number",
        "finding_class_normalized",
        "image_path",
        "image_relpath",
    }

    missing_temporal_columns = sorted(
        required_temporal_columns
        - set(temporal_manifest.columns)
    )

    missing_labelled_columns = sorted(
        required_labelled_columns
        - set(labelled_manifest.columns)
    )

    if missing_temporal_columns:
        raise KeyError(
            "Temporal context metadata is missing columns: "
            f"{missing_temporal_columns}"
        )

    if missing_labelled_columns:
        raise KeyError(
            "Labelled-frame metadata is missing columns: "
            f"{missing_labelled_columns}"
        )

    verified_frames = (
        labelled_manifest[
            [
                "video_key",
                "frame_number",
            ]
        ]
        .drop_duplicates()
        .rename(
            columns={
                "frame_number":
                    "context_frame_number"
            }
        )
        .assign(
            context_is_verified=True
        )
    )

    verified_findings = (
        labelled_manifest[
            [
                "video_key",
                "frame_number",
                "finding_class_normalized",
            ]
        ]
        .drop_duplicates()
        .rename(
            columns={
                "frame_number":
                    "context_frame_number"
            }
        )
        .assign(
            context_is_verified_same_finding=True
        )
    )

    image_path_consistency = (
        labelled_manifest
        .groupby(
            [
                "video_key",
                "frame_number",
            ],
            sort=False,
            observed=True,
        )[
            [
                "image_path",
                "image_relpath",
            ]
        ]
        .nunique(
            dropna=False
        )
    )

    inconsistent_image_mask = (
        image_path_consistency
        .gt(1)
        .any(axis=1)
    )

    if inconsistent_image_mask.any():

        inconsistent_frames = (
            image_path_consistency.index[
                inconsistent_image_mask
            ]
            .tolist()
        )

        raise ValueError(
            "Conflicting official image paths found for "
            f"verified frames: {inconsistent_frames}"
        )

    official_frame_images = (
        labelled_manifest
        .sort_values(
            [
                "video_key",
                "frame_number",
                "finding_class_normalized",
            ],
            kind="stable",
        )
        .groupby(
            [
                "video_key",
                "frame_number",
            ],
            as_index=False,
            sort=False,
            observed=True,
        )
        .agg(
            context_official_image_path=(
                "image_path",
                "first",
            ),

            context_official_image_relpath=(
                "image_relpath",
                "first",
            ),

            context_verified_labels=(
                "finding_class_normalized",
                lambda values:
                    "|".join(
                        sorted(
                            values
                            .dropna()
                            .astype("string")
                            .unique()
                            .tolist()
                        )
                    ),
            ),
        )
        .rename(
            columns={
                "frame_number":
                    "context_frame_number"
            }
        )
    )

    result = (
        temporal_manifest
        .merge(
            verified_frames,
            on=[
                "video_key",
                "context_frame_number",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            verified_findings,
            on=[
                "video_key",
                "context_frame_number",
                "finding_class_normalized",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            official_frame_images,
            on=[
                "video_key",
                "context_frame_number",
            ],
            how="left",
            validate="many_to_one",
        )
        .assign(
            context_is_verified=lambda data:
                data[
                    "context_is_verified"
                ]
                .fillna(False)
                .astype("boolean"),

            context_is_verified_same_finding=lambda data:
                data[
                    "context_is_verified_same_finding"
                ]
                .fillna(False)
                .astype("boolean"),
        )
    )

    invalid_target_mask = (
        result["is_target"]
        .astype("boolean")
        &
        (
            ~result[
                "context_is_verified_same_finding"
            ]
            | result[
                "context_official_image_path"
            ].isna()
        )
    )

    if invalid_target_mask.any():

        invalid_targets = (
            result.loc[
                invalid_target_mask,
                [
                    "finding_segment_id",
                    "video_key",
                    "context_frame_number",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Temporal targets could not be matched to "
            "their official verified images: "
            f"{invalid_targets}"
        )

    annotation_status = np.select(
        [
            result[
                "is_target"
            ].astype(bool),

            result[
                "context_is_verified_same_finding"
            ].astype(bool),

            result[
                "context_is_verified"
            ].astype(bool),
        ],
        [
            "verified_target",
            "verified_same_finding",
            "verified_other_finding",
        ],
        default="unlabelled_context",
    )

    return (
        result
        .assign(
            context_annotation_status=(
                pd.Series(
                    annotation_status,
                    index=result.index,
                    dtype="string",
                )
            ),

            context_image_path=(
                result[
                    "context_official_image_path"
                ]
            ),

            context_image_relpath=(
                result[
                    "context_official_image_relpath"
                ]
            ),

            context_image_source=(
                pd.Series(
                    np.where(
                        result[
                            "context_official_image_path"
                        ].notna(),
                        "official_labelled_image",
                        "video_extraction_required",
                    ),
                    index=result.index,
                    dtype="string",
                )
            ),
        )
    )

temporal_manifest = (
    add_temporal_context_metadata(
        temporal_manifest=(
            temporal_manifest
        ),
        labelled_manifest=(
            labelled_frame_manifest
        ),
    )
)


display(
    temporal_manifest[
        "context_image_source"
    ].value_counts(
        dropna=False
    )
)

temporal_metadata_dir = (
    Path(
        DIRS[
            "temporal_frames_dir"
        ]
    )
    .parent
    / "manifests"
)


### 27. Prepare only missing temporal frames for extraction

In [ ]:
TEMPORAL_EXTRACTION_REQUEST_COLUMNS = [
    "video_key",
    "opencv_frame_index",
    "video_path",
    "output_path",
]


def build_temporal_output_path(
    video_key,
    opencv_frame_index,
    dirs,
    config,
):
    """
    Builds the deterministic output path for one
    temporal context frame.
    """

    extension = (
        str(
            config[
                "temporal_context_image_format"
            ]
        )
        .strip()
        .lower()
        .lstrip(".")
    )

    if extension not in {
        "jpg",
        "jpeg",
        "png",
    }:
        raise ValueError(
            "Unsupported temporal context image format: "
            f"{extension}"
        )

    return (
        Path(
            dirs[
                "temporal_frames_dir"
            ]
        )
        / str(video_key)
        / (
            f"frame_"
            f"{int(opencv_frame_index):08d}"
            f".{extension}"
        )
    )


def build_temporal_extraction_requests(
    temporal_manifest,
    video_manifest,
    dirs,
    config,
):
    """
    Builds one extraction request per unique temporal
    context frame requiring an image from the source video.

    This function plans extraction but performs no disk I/O.
    """

    required_temporal_columns = {
        "video_key",
        "opencv_frame_index",
        "context_image_path",
        "context_image_source",
    }

    required_video_columns = {
        "video_key",
        "video_path",
    }

    missing_temporal_columns = sorted(
        required_temporal_columns
        - set(temporal_manifest.columns)
    )

    missing_video_columns = sorted(
        required_video_columns
        - set(video_manifest.columns)
    )

    if missing_temporal_columns:
        raise KeyError(
            "Temporal extraction planning is missing "
            f"columns: {missing_temporal_columns}"
        )

    if missing_video_columns:
        raise KeyError(
            "Video manifest is missing columns: "
            f"{missing_video_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    extraction_mask = (
        temporal_manifest[
            "context_image_source"
        ]
        .astype("string")
        .eq(
            "video_extraction_required"
        )
        .fillna(False)
    )

    inconsistent_mask = (
        extraction_mask
        & temporal_manifest[
            "context_image_path"
        ].notna()
    )

    if inconsistent_mask.any():
        raise ValueError(
            "Some frames are marked as requiring video "
            "extraction but already have an image path."
        )

    requests = (
        temporal_manifest.loc[
            extraction_mask,
            [
                "video_key",
                "opencv_frame_index",
            ],
        ]
        .drop_duplicates()
        .reset_index(
            drop=True
        )
    )

    if requests.empty:
        return pd.DataFrame(
            columns=(
                TEMPORAL_EXTRACTION_REQUEST_COLUMNS
            )
        )

    frame_indices = pd.to_numeric(
        requests[
            "opencv_frame_index"
        ],
        errors="coerce",
    )

    invalid_index_mask = (
        frame_indices.isna()
        | ~np.isfinite(
            frame_indices.astype("float64")
        )
        | frame_indices.lt(0)
        | frame_indices.mod(1).ne(0)
    )

    if invalid_index_mask.any():
        raise ValueError(
            "Temporal extraction requests contain "
            "invalid OpenCV frame indices."
        )

    requests = (
        requests
        .assign(
            opencv_frame_index=(
                frame_indices.astype("Int64")
            )
        )
        .merge(
            video_manifest[
                [
                    "video_key",
                    "video_path",
                ]
            ],
            on="video_key",
            how="left",
            validate="many_to_one",
            indicator=True,
        )
    )

    missing_video_mask = (
        requests[
            "_merge"
        ]
        .ne("both")
        | requests[
            "video_path"
        ].isna()
    )

    if missing_video_mask.any():
        missing_video_keys = (
            requests.loc[
                missing_video_mask,
                "video_key",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise KeyError(
            "Videos required for temporal extraction "
            "are missing: "
            f"{missing_video_keys}"
        )

    requests = (
        requests
        .drop(
            columns="_merge"
        )
        .assign(
            output_path=lambda data: [
                str(
                    build_temporal_output_path(
                        video_key=video_key,
                        opencv_frame_index=(
                            opencv_frame_index
                        ),
                        dirs=dirs,
                        config=config,
                    )
                )
                for video_key, opencv_frame_index
                in zip(
                    data["video_key"],
                    data["opencv_frame_index"],
                )
            ]
        )
        .loc[
            :,
            TEMPORAL_EXTRACTION_REQUEST_COLUMNS,
        ]
        .sort_values(
            [
                "video_key",
                "opencv_frame_index",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    if requests[
        "output_path"
    ].duplicated().any():
        raise RuntimeError(
            "Multiple temporal frames resolve to the "
            "same output path."
        )

    return requests


temporal_extraction_requests = (
    build_temporal_extraction_requests(
        temporal_manifest=temporal_manifest,
        video_manifest=video_manifest,
        dirs=DIRS,
        config=CONFIG,
    )
)


print(
    "Unique temporal frames requiring extraction:",
    f"{len(temporal_extraction_requests):,}",
)

display(
    temporal_extraction_requests.head()
)

### 28. Run temporal frame extraction

In [ ]:
TEMPORAL_EXTRACTION_RESULT_COLUMNS = [
    "video_key",
    "opencv_frame_index",
    "extracted_image_path",
    "extraction_status",
    "extraction_error",
]


def get_temporal_image_write_parameters(
    output_path,
    config,
):
    """
    Returns the OpenCV encoding parameters appropriate
    for the requested output format.
    """

    extension = output_path.suffix.lower()

    if extension in {
        ".jpg",
        ".jpeg",
    }:
        quality = int(
            config[
                "temporal_context_jpeg_quality"
            ]
        )

        if not 0 <= quality <= 100:
            raise ValueError(
                "JPEG quality must be between 0 and 100."
            )

        return [
            cv2.IMWRITE_JPEG_QUALITY,
            quality,
        ]

    if extension == ".png":
        compression = int(
            config.get(
                "temporal_context_png_compression",
                3,
            )
        )

        if not 0 <= compression <= 9:
            raise ValueError(
                "PNG compression must be between 0 and 9."
            )

        return [
            cv2.IMWRITE_PNG_COMPRESSION,
            compression,
        ]

    raise ValueError(
        f"Unsupported temporal image format: {extension}"
    )


def extract_temporal_frames_for_video(
    requests,
    config,
):
    """
    Extracts requested frames from exactly one video.

    Side effects:
        Creates output directories.
        Writes JPEG or PNG files to persistent storage.

    Returns:
        One extraction-result record per requested frame.
    """

    if requests.empty:
        return pd.DataFrame(
            columns=(
                TEMPORAL_EXTRACTION_RESULT_COLUMNS
            )
        )

    required_columns = {
        "video_key",
        "video_path",
        "opencv_frame_index",
        "output_path",
    }

    missing_columns = sorted(
        required_columns
        - set(requests.columns)
    )

    if missing_columns:
        raise KeyError(
            "Temporal extraction requests are missing "
            f"columns: {missing_columns}"
        )

    if requests[
        "video_path"
    ].isna().any():
        raise ValueError(
            "Temporal extraction requests contain "
            "missing video paths."
        )

    video_paths = (
        requests[
            "video_path"
        ]
        .astype("string")
        .unique()
    )

    if len(video_paths) != 1:
        raise ValueError(
            "A video extraction batch must reference "
            "exactly one source video."
        )

    if requests.duplicated(
        subset=[
            "video_key",
            "opencv_frame_index",
        ]
    ).any():
        raise ValueError(
            "Duplicate frame requests found within "
            "the same video batch."
        )

    ordered_requests = (
        requests
        .sort_values(
            "opencv_frame_index",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    video_path = Path(
        video_paths[0]
    )

    capture = cv2.VideoCapture(
        str(video_path)
    )

    if not capture.isOpened():
        capture.release()

        raise RuntimeError(
            f"Unable to open video: {video_path}"
        )

    records = []

    try:

        for request in ordered_requests.itertuples(
            index=False
        ):
            frame_index = int(
                request.opencv_frame_index
            )

            output_path = Path(
                request.output_path
            )

            output_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            if (
                output_path.is_file()
                and output_path.stat().st_size > 0
            ):
                success = True
                status = "reused_existing"
                error_message = None

            else:
                seek_success = capture.set(
                    cv2.CAP_PROP_POS_FRAMES,
                    frame_index,
                )

                if not seek_success:
                    success = False
                    status = "seek_failed"
                    error_message = (
                        "OpenCV could not seek to the "
                        f"requested frame {frame_index}."
                    )

                else:
                    readable, frame = capture.read()

                    if not readable or frame is None:
                        success = False
                        status = "read_failed"
                        error_message = (
                            "OpenCV could not decode the "
                            f"requested frame {frame_index}."
                        )

                    else:
                        write_parameters = (
                            get_temporal_image_write_parameters(
                                output_path=output_path,
                                config=config,
                            )
                        )

                        try:
                            write_success = cv2.imwrite(
                                str(output_path),
                                frame,
                                write_parameters,
                            )

                        except cv2.error as error:
                            write_success = False
                            error_message = str(error)

                        success = bool(
                            write_success
                            and output_path.is_file()
                            and output_path.stat().st_size > 0
                        )

                        if success:
                            status = "extracted"
                            error_message = None

                        else:
                            status = "write_failed"

                            if error_message is None:
                                error_message = (
                                    "OpenCV did not write a "
                                    "valid output image."
                                )

            records.append(
                {
                    "video_key":
                        request.video_key,

                    "opencv_frame_index":
                        frame_index,

                    "extracted_image_path":
                        (
                            str(output_path)
                            if success
                            else None
                        ),

                    "extraction_status":
                        status,

                    "extraction_error":
                        error_message,
                }
            )

    finally:
        capture.release()

    return pd.DataFrame.from_records(
        records,
        columns=(
            TEMPORAL_EXTRACTION_RESULT_COLUMNS
        ),
    )

extracted_temporal_frames = pd.DataFrame(
    columns=(
        TEMPORAL_EXTRACTION_RESULT_COLUMNS
    )
)


if CONFIG[
    "temporal_context_extract_frames"
]:

    extracted_tables = [
        extract_temporal_frames_for_video(
            requests=video_requests,
            config=CONFIG,
        )

        for _, video_requests
        in tqdm(
            temporal_extraction_requests.groupby(
                "video_key",
                sort=False,
                observed=True,
            ),
            desc="Extracting temporal evidence",
        )
    ]


    extracted_temporal_frames = (
        pd.concat(
            extracted_tables,
            ignore_index=True,
        )

        if extracted_tables

        else pd.DataFrame(
            columns=(
                TEMPORAL_EXTRACTION_RESULT_COLUMNS
            )
        )
    )


    duplicate_result_mask = (
        extracted_temporal_frames.duplicated(
            subset=[
                "video_key",
                "opencv_frame_index",
            ],
            keep=False,
        )
    )


    if duplicate_result_mask.any():
        raise RuntimeError(
            "Temporal extraction produced duplicate "
            "frame-result records."
        )


    # Makes the cell safe to rerun without creating
    # extracted_image_path_x and extracted_image_path_y.
    existing_extraction_columns = [
        column
        for column in [
            "extracted_image_path",
            "extraction_status",
            "extraction_error",
        ]
        if column in temporal_manifest.columns
    ]


    temporal_manifest = (
        temporal_manifest
        .drop(
            columns=existing_extraction_columns
        )
        .merge(
            extracted_temporal_frames,
            on=[
                "video_key",
                "opencv_frame_index",
            ],
            how="left",
            validate="many_to_one",
        )
    )


    # Prefer the official labelled image when available,
    # otherwise use the frame extracted from video.
    final_context_paths = (
        temporal_manifest[
            "context_official_image_path"
        ]
        .combine_first(
            temporal_manifest[
                "extracted_image_path"
            ]
        )
        .combine_first(
            temporal_manifest[
                "context_image_path"
            ]
        )
    )


    context_sources = np.select(
        [
            temporal_manifest[
                "context_official_image_path"
            ].notna(),

            final_context_paths.notna(),
        ],
        [
            "official_labelled_image",
            "extracted_from_video",
        ],
        default="missing",
    )


    temporal_manifest = (
        temporal_manifest
        .assign(
            context_image_path=(
                final_context_paths
            ),

            context_image_source=(
                pd.Series(
                    context_sources,
                    index=temporal_manifest.index,
                    dtype="string",
                )
            ),
        )
    )


    display(
        extracted_temporal_frames[
            "extraction_status"
        ].value_counts(
            dropna=False
        )
    )


def path_to_storage_relpath_or_missing(
    path,
):
    """
    Converts one absolute storage path into a relative path
    while preserving missing values.
    """

    if pd.isna(path):
        return pd.NA

    return path_relative_to_storage(
        path
    )


temporal_manifest[
    "context_image_relpath"
] = (
    temporal_manifest[
        "context_image_path"
    ]
    .map(
        path_to_storage_relpath_or_missing
    )
    .astype("string")
)


display(
    temporal_manifest[
        "context_image_source"
    ].value_counts(
        dropna=False
    )
)

### 29. Validate and persist final temporal artifacts


In [ ]:
temporal_metadata_dir = (
    Path(
        DIRS[
            "temporal_frames_dir"
        ]
    )
    .parent
    / "manifests"
)


temporal_metadata_dir.mkdir(
    parents=True,
    exist_ok=True,
)


temporal_manifest_path = (
    temporal_metadata_dir
    / "temporal_manifest.parquet"
)


temporal_extraction_report_path = (
    temporal_metadata_dir
    / "temporal_extraction_report.parquet"
)


temporal_extraction_failure_path = (
    temporal_metadata_dir
    / "temporal_extraction_failures.parquet"
)

### 29. Image quality metrics

In [ ]:
def compute_image_qc(
    image_path,
):
    """
    Computes basic image-quality proxies for one capsule frame.

    These metrics are not clinical quality labels.
    """

    image = cv2.imread(
        str(
            image_path
        )
    )

    if image is None:

        return {
            "image_path":
                str(
                    image_path
                ),

            "image_readable":
                False,

            "image_width":
                np.nan,

            "image_height":
                np.nan,

            "brightness_mean":
                np.nan,

            "contrast_std":
                np.nan,

            "blur_laplacian":
                np.nan,

            "underexposed_fraction":
                np.nan,

            "overexposed_fraction":
                np.nan,

            "specular_fraction":
                np.nan,
        }

    height, width = (
        image.shape[:2]
    )

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY,
    )

    hsv = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2HSV,
    )

    saturation = (
        hsv[:, :, 1]
    )

    value = (
        hsv[:, :, 2]
    )

    return {
        "image_path":
            str(
                image_path
            ),

        "image_readable":
            True,

        "image_width":
            width,

        "image_height":
            height,

        "brightness_mean":
            float(
                gray.mean()
                / 255.0
            ),

        "contrast_std":
            float(
                gray.std()
                / 255.0
            ),

        "blur_laplacian":
            float(
                cv2.Laplacian(
                    gray,
                    cv2.CV_64F,
                ).var()
            ),

        "underexposed_fraction":
            float(
                np.mean(
                    gray < 10
                )
            ),

        "overexposed_fraction":
            float(
                np.mean(
                    gray > 245
                )
            ),

        "specular_fraction":
            float(
                np.mean(
                    (value > 245)
                    &
                    (saturation < 40)
                )
            ),
    }